# `C15` — Randomised fuzz across the parameter space

Scenario **C15** of the Track A certification ladder (card `OMIE-CERT-C15-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C15; plan [`scenario-ladder-plan.md`](../../docs/omiecid/scenario-ladder-plan.md) §C15, §The invariant battery and §The capacity-validity precondition). Oracle kind: **invariant / distributional**.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. A *draw* is one whole trading session generated from one integer seed; a *cell* is one point of the regime grid the draws sweep; a *variation* is C15a, C15b or C15c. Every pass/fail assertion is an **exact integer** comparison (order ids, prices in ticks of 0.01 EUR/MWh, quantities in ticks of 0.1 MW, counts, statuses); euro figures are shown, never asserted.

## 1. Purpose

**Purpose.** This is the scenario that converts "the simulator is correct" from a list of anecdotes
into a defensible claim, and it is the direct answer to the request for randomly generated datasets
at scale. Every scenario above fixes a construction and randomises within it; C15 randomises the
construction itself, precisely to reach the interaction bugs that only appear when an iceberg
regenerates into a fill-or-kill pre-check during a tie.

*(Verbatim from `scenario-ladder.md` §C15.)*

**Certifies** (card): §4.5, §6.1, §6.2 jointly, under interaction; §6.6 at scale via M6. **Known result** (card): zero exceptions across I1–I9 and M1–M6, plus a pre-registered closed-form band on aggregate matched volume (a sanity signal, never a pass criterion on its own). **Pass** (card): zero exceptions across every draw — *one violation is a certification failure, not a statistic*. **On failure:** shrink to the minimal violating session and freeze it as a regression tape under `tests/omiecid/fixtures/cert/`. Common to every scenario: eager and jitted agree; integer-tick assertions; the full battery runs; I8 capacity headroom holds on all five ceilings (a breach makes the draw **INVALID, not failed**: it is re-drawn and the discard rate is reported); I9 holds; every quoted passage matches its source.

**What this notebook asserts, per valid draw** (section 5 states each part; section 7 evaluates and records it):

| | assertion | judged by |
|---|---|---|
| **B** | the battery RB, I1–I9 on the jitted run: **no check returns `fail`**. `pass` and `not_applicable` are both acceptable (a run with no fill legitimately makes several checks `not_applicable`); I8 `invalid` discards the draw; I9's "construction implies a trade" is taken from the reference walk **W** | `run_battery` |
| **W** | an **independent price–time reference walk** over the session's tape (visible and hidden slices, §4.2 slice activation, §4.2.1 Rules 1–4 including the incoming iceberg as a chain of instances (Rule 2 *Incoming iceberg* items (1)–(6) and Rule 3 *ICE aggressor*, amended 2026-10-20) and eager activation, §5.2, §6.1, §6.2, OQ-11, OQ-31, OQ-32 (3b), OQ-33 and OQ-34 (their 2026-10-20 blocks)) predicts the engine's trade sequence **row for row** — buyer id, seller id, price, quantity, submitting order — plus every order's cancelled quantity (with its §5.1 cause) and its remaining visible + hidden quantity, and, **after every submission**, the book every order is shown in: its price, its visible quantity and its hidden quantity, so that every iceberg with remaining quantity **shows a slice** (finding F6). Strict: no activation reading is left open. Run on **every** session, not only those holding icebergs, because the battery's I4 reads only the visible book and cannot judge hidden-slice priority (review D3-08) | section 5 `RefWalk`, `judge_reference` |
| **M** | the metamorphic relations: M4 on every fill-or-kill order of every draw (no extra run); one of M1, M2, M3, M5, M6 per draw in rotation (finding F10), each on a variant run that must itself pass **B** and **W** | `check_m1` … `check_m6` |
| **P** | the dataset precondition: the 13-column integer layout, §2.3 bounds, the derived iceberg peak rule, the §13.6 sign rule, the slice margin $\lceil \mathit{total}/\mathit{peak} \rceil \le 64$, and the draw's analytic capacity bounds inside its cell's envelope | section 4 `precondition` |
| **E** | eager == jitted (`compare_records`) on the first valid 32-order draw of every order-type mix; a mix with no eager comparison is a **failed** `eager_jit` assertion | section 7 |

**Aggregate assertions:** the section-6 wiring; the five documented ceilings and the pressure configurations; every requested draw ran (valid = requested per variation); every cell of every variation has a valid draw (the sweep is systematic); every relation applied at least `MIN_RELATION_APPLICATIONS` times per variation where the mix allows it; M6 exercised at least one self-match per variation; on every variation whose mixes hold icebergs, the amended incoming-iceberg branch was exercised (finding F21); the shrinker reproduced every failure it shrank. **Reported, not asserted:** the closed-form band on aggregate matched volume (section 5), the discard rate per variation and per pressure level, the capacity peaks, the number of slices W activated and of book states checked (finding F6), the coverage counts of the incoming-iceberg branch (finding F21), and the wall clock.

**K2 findings** (recorded here; none blocks the expectation):

- **F1 — spec version.** The card names `market-spec.md` v2.12; the file is now **v2.14** (v2.13: OQ-31, §4.2.1 Rule 3 and §6.2 item 7; v2.14: §4.2.1 Rule 2's re-ranking of a regenerated slice and its *Incoming iceberg* items (1)–(6), Rule 4's eager slice activation, the §5.4 ICEwPI and peak rejections), **amended 2026-10-20** under the kept v2.14 token (`OMIE-SPEC-ICE-INCOMING-AMEND-1`, user ruling 2026-10-20; **not additive**): Rule 2 *Incoming iceberg* items (2), (4), (6) and its classification rewritten, the superseded sentences deleted, and the first sub-bullet of Rule 3 *ICE aggressor* rewritten; OQ-33 revised, OQ-34 resolved, OQ-35 filed. The OQ-32 2026-10-08 scope correction (3b), quoted in section 2, is queued for the next spec revision; W follows it. Every quote is checked against the current file at run time.
- **F2 — the two pressure axes are built here.** The card asks for book depth and iceberg-slot pressure as **independent** axes (review §5 item 3). `fuzz_grid` has neither: its grid is mix × crossing probability × tie density × length, and every draw runs at the default ceilings, where at most 160 icebergs can ever exist against a ceiling of 1 024. The notebook adds both axes as **configured engine ceilings** with an 80 % generation margin on the matching envelope axis, each independently: book depth $D \in \{100, 60\}$ (margin 80 / 48 resting per side) and iceberg slots $M \in \{1024, 64\}$ (margin 800 / 51 concurrent icebergs). The two are independent fields of `OMIEMarketConfig` and of `CapacityEnvelope`. A cell whose session length exceeds twice its resting margin is **structurally infeasible** for the same reason the length axis stops at 160 (a tape carries no cancellations), so $D = 60$ is crossed with lengths 32 and 96 only.
- **F3 — C15c's weighting.** The generator has no weight parameter. C15c sweeps C15b's cells with weights 3 for mixes holding both FOK and ICE (`non_ioc_fok_ice`, `all_icewpi`), 2 for `non_ioc_fok`, 1 otherwise. C15c is opt-in (`VARIATIONS` containing `c`) and is not in the gate (card: nightly).
- **F4 — C15a's "short sessions"** are read as $n = 32$, with mixes `non` and `non_ioc`, every crossing probability, tie density and pressure level.
- **F5 — the judging limit, and why W exists (restated at r4).** The battery's I4 reads the visible book before each submission, so it cannot order hidden or regenerated slices (review D3-08: *"The fix belongs at a scenario oracle, not in I4."*). W is that oracle. It is the real matching algorithm (Matching Algorithm §5.1.3: the incoming order is matched while it remains executable), so the §4.2.1 Rule-3 hop is not modelled as a separate event: its trades are the same rows, at the iceberg slice's own price (OQ-31). An **incoming iceberg** of base $b$, peak $P$ and increment $d$ ($d = 0$ for a standard iceberg) follows v2.14 §4.2.1 Rule 2 *Incoming iceberg* as amended on 2026-10-20 (OQ-33 revised, OQ-34 resolved), for **every** $d$: it aggresses with its total quantity as a **chain of instances** of size $P$, the last possibly smaller (items (1), (2)); instance $k$ trades only at prices its own limit $b + k\,d$ crosses, and the sweep **stops at the first instance that cannot trade**, even if $b$ would still cross (item (6)); every trade prints at the resting entry's price (item (3); Rule 3 *ICE aggressor*); and a row ends where a resting entry ends **or** where an incoming instance ends, so there is one row per (resting entry, incoming instance) pair (item (2)). With $M$ the quantity swept, a remainder $R > 0$ rests as an iceberg that keeps its place in the ladder: instance $k_e = \lfloor M/P \rfloor$ at $b + k_e\,d$, showing $v_1 = \min(P - u, R)$ with $u = M - k_e P$ and keeping $R - v_1$ hidden, with its arrival number (item (4); §5.2); its later slices continue the ladder at $b + (k_e + 1)\,d$, $b + (k_e + 2)\,d$, … (no restart at slice 0). It is never IOC: nothing of it is cancelled (item (5)). The display $v_1$ after a part-used instance ($u > 0$) is, per item (4), derived from OMIE data rather than stated by the sources; W pins it because the rule states it. Rows are pinned by buyer and seller order id, price, quantity and the **submitting order's id**; which leg the engine *labels* passive is not asserted (as in C08, finding F3 there). The rule's *Simulator limit* (rows left unsplit on a trade ring without room) is never reached on a valid draw: every row ends at the end of a resting entry or of an incoming instance, each such unit ends once, so a session's rows are bounded by its trading units, which the envelope keeps at 80 below the ring's 100.
- **F6 — slice activation: eager, queued at the event, and shown (tightened at r3).** The r1/r2 expectation accepted a *lazy* reading for a slice activated in one submission and untouched by its end. That tolerance is **withdrawn**: v2.14 §4.2.1 Rule 4 (*Slice activation timing*) says *"Activation is **eager**"* and *"between matching processes, an iceberg with remaining quantity always shows a slice"*; the OQ-20 correction note and its 2026-10-08 answer say the same and route the *"slice shown"* outcome to this finding. W now (i) activates the next slice the moment the previous one is used up, inside that matching process; (ii) gives it its queue place **at that event** (E2; OQ-32 (3b)): a new number of the one global sequence, so it is after the order whose matching process activated it, behind every order then resting at its price, and ahead of every order that arrives later (§5.2, §6.1); and (iii) asserts, on the engine's post-submission book after **every** submission, that each order W holds is shown at W's price with W's visible quantity and holds W's hidden quantity, nothing else being shown or hidden — in particular, that **no iceberg with remaining quantity is left without a shown slice** (recorded as `slice_shown`). The queue place is judged by its consequence, the strict trade sequence: a slice activated in an earlier matching process and served after a later arrival at its price is a `trade_sequence` mismatch. The engine's stored stamp values (seconds, sub-second rank) are **not** asserted: their encoding is engine-internal; only the order they induce is observable here.
- **F7 — M1 needs a soundness condition in a whole session.** Swapping two tied orders that are marketable on arrival can legitimately change aggregate matched volume (an IOC and a NON competing for the same maker leave different books behind). M1 is applied only to tie groups (consecutive rows with one tape timestamp) whose members, in W, trade nothing on their own submission and contain no opposite-side pair that could cross by price. Such a group enters an identical book in any order and leaves the same price levels with the same depth, differing only in queue order at equal price; since every slice ladder is monotone (§13.6 sign rule), each later aggressor's matched quantity — $\min(Q,\ \text{crossing units})$ — is unchanged. An incoming iceberg's instance-limited sweep (F5) is covered too: every resting ladder only worsens, so a price level is used up before a worse one is touched, the best price met after $M$ units is a function of the depth at each price alone, and so is where its instance limits stop it. The permutation is the group's reversal.
- **F8 — M2, M3 and M5 run on a prefix ending at the chosen aggressor**, so the book it meets is static and nothing after it can blur the comparison. The aggressor is the first NON or IOC order that trades on its own submission in W (M5: the first such whose quantity divides by 2 or 3). FOK is excluded from M2 (a larger fill-or-kill can legitimately be killed) and icebergs from M3 and M5 (not their subject). M2 alternates *size doubled* and *limit tightened by 1–3 ticks* across draws.
- **F9 — M4 and M6.** M4 runs on every FOK of every draw, on the draw's own run. M6 relabels owners $1 \to 2 \to 3 \to 4 \to 1$ on the whole tape; the generator gives some orders a common owner, so self-matches occur, and their presence in at least one M6 application per variation is asserted (§6.6 at scale).
- **F10 — the relation budget.** One relation per draw, rotating through M1, M2, M3, M5, M6 and falling through to the next when one is not applicable (`RELATIONS_PER_DRAW` raises it). Each costs one full or two prefix runs; running all five on every draw would multiply the gate's runtime about fivefold.
- **F11 — I9 non-vacuity.** "Where the construction implies a trade" is read as: W predicts at least one trade on the draw (or variant).
- **F12 — the band uses the generator's size law**, which is not exposed in `Draw.params` (lognormal $\mu = 1.6$, $\sigma = 0.7$, times 10, rounded and clipped to 1–600 ticks, read from `omie_tape_synth.py`). The empirical mean order size is reported beside it. The band is reported and **never asserted**.
- **F13 — eager coverage.** An eager run of a 160-order session does not fit the gate. Eager is compared on the first valid 32-order draw of each mix (`EAGER_PER_MIX`), and its absence is a failure.
- **F14 — the shrinker** bisects on the tape prefix, then greedily deletes single earlier rows while the **same check** still fails, within an evaluation budget. The result is frozen to `tests/omiecid/fixtures/cert/C15_shrunk_seed<k>.npz` (or `$OMIE_CERT_FIXTURE_DIR`), and recorded with `record_case`, so the harness freezes it as section 9's smallest case.
- **F15 — the `P3-PERF-REGEN-MASK-1` debt.** The card says the frozen fixtures **plus one real-contract golden** discharge it. A real-contract golden needs raw OMIE data the notebook cannot assume exists (the repo rule is not to assume any specific stock or time period is present), so it is not produced here; only the synthetic half is. Flagged for the orchestrator.
- **F16 — acceptance.** Every generated row satisfies §2.3, the derived iceberg rule $0 < \mathit{peak} \le \mathit{quantity}$ (C01) and the §13.6 sign rule, and no other §5.4 cause applies to a synthetic open contract, so every order must be accepted.
- **F17 — record size.** Every draw is recorded in full but compactly: one string per draw (seed, content digest, the status of every check, relation, reference walk, trade count and volume), grouped per cell; violations and mismatches are recorded in full for failing draws only. A failing draw's book check records the number of submissions after which the book differed, all their order ids, and the first difference in full; every `slice_shown` breach is recorded.
- **F18 — retired at r4.** It labelled the incoming-ICEwPI base-limit sweep, and the restart of the rested remainder's drift at slice 0, a modelling choice (OQ-34, then open), and pinned it. OQ-34 is now resolved by the user's ruling of 2026-10-20 and the rule is §4.2.1 Rule 2 items (4) and (6), so W pins a rule, not a choice (F5). The section-2 quotes of OQ-33's *Ruling: one trade per counter bid* block, of OQ-34's *The simulator's convention* block and of the deleted §4.2.1 sentences are retired.
- **F19 — r3 K2 revision (`OMIE-CERT-RERUN-2`, verdict DEFECT-FOUND).** Changed: W's incoming-iceberg branch (from peak-first, one row per incoming slice, its remainder carrying its drifted slice price and a new number, to v2.14 Rule 2 as in F5 and F18); W's slice queueing made strict at the activation event (E2) with the F6 lazy tolerance removed; the book check of F6 added. Unchanged: the battery, the relations and their variants, the precondition, the grid, the band, the eager rule, the shrinker and every other assertion. The relations' aggressor choice reads W's trades, so their selection on draws holding incoming icebergs may differ from r2.
- **F20 — r4 K2 revision (`OMIE-CERT-RERUN-3`).** Changed: W's incoming-iceberg branch, from r3's total sweep at the base limit with one row per resting entry and a remainder showing $\min(P, R)$ at $b$ with its drift restarted at slice 0, to the instance chain of F5 (per-instance limits, the stop at the first instance that cannot trade, rows split at incoming instance boundaries, the remainder continuing as instance $k_e$ at $b + k_e\,d$ showing $v_1$, later slices continuing the ladder), for every $d$; F5 restated; F18 retired; F7's soundness argument extended to the instance-limited sweep; section 2's incoming-iceberg quotes replaced by the amended §4.2.1 Rule 2 items (2)–(6), Rule 3 *ICE aggressor* and the 2026-10-20 blocks of OQ-33 and OQ-34 (with OQ-35 for the price bound); the coverage assertion F21 added. Standard incoming icebergs ($d = 0$) change too: their rows now split at instance boundaries, and a part-used instance shows $P - u$, not $\min(P, R)$. Unchanged: the battery, the relations and their variants, the precondition, the grid, the band, the eager rule, the shrinker, E2 queueing, F6's book check and every other assertion. The relations' aggressor choice reads W's trades (M2, M3 and M5 take NON or IOC aggressors only), so it may differ from r3 only on draws whose incoming icebergs now trade differently.
- **F21 — the amended branch is exercised (non-vacuity).** W counts, on its own walk of each valid draw: an incoming ICEwPI ($d \ne 0$) that trades on entry; an incoming iceberg stopped by its instance limit while $b$ would still cross; a row ended at an incoming instance boundary; a remainder left with $u > 0$; and a later print on a remainder resting off its base ($k_e > 0$, $d \ne 0$), i.e. the ladder continued. On every variation whose mixes hold icebergs, the number of draws with each must be at least 1 (asserted, in the spirit of I9); all counts are reported. A K2 walk of the 2,200 primary seeds (generator and W only, no engine; replacement seeds unused, since no generation breach occurred) found, of C15b's 2,000 draws, 241 with a trading incoming ICEwPI, 160 with an instance-limit stop, 605 with an instance-split row, 378 with a remainder at $u > 0$ (319 of them standard, 98 ICEwPI, some both) and 146 with a continued-ladder print (122 with a continued slice activated); C15a holds no icebergs. The engine's I8 discards, unknown at K2, can only lower these.


## 2. The rule, quoted

The governing passages, quoted **verbatim** (byte-copied) and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. Text from the regulatory corpus is shown after them as **citations**, rendered but not machine-checked.

How the quotes ground the expectation of section 5:

- **What each restriction does (§4.5, §6.2 items 4–5, §4.2).** NON rests its residual, IOC cancels it, FOK fills fully or is cancelled, an iceberg shows only its peak and rests its hidden quantity. Hence W's per-order outcome: remaining visible + hidden, and the cancelled quantity with its §5.1 cause (`ioc_remainder`, `fok_kill`).
- **Price–time priority and maker pricing (§6.1, §6.2 items 1–3, 6).** W walks the opposite side best price first, earliest timestamp first, and prints each trade at the resting order's limit.
- **Slices (§4.2, §5.2, §4.2.1 Rules 2–4, §6.2 item 7, OQ-31, OQ-32 (3b)).** A new slice is a new-timestamp entry at its own (drifted) price, ranked as a new entry and matched inside the same submission, queued at its activation event; the iceberg sets the price, including at the Rule-3 hop. Hence W's activation, its queue place and the price of every slice trade.
- **The incoming iceberg (§4.2.1 Rule 2 *Incoming iceberg* items (1)–(6) and Rule 3 *ICE aggressor*, amended 2026-10-20; the 2026-10-20 blocks of OQ-33 and OQ-34).** Total quantity as a chain of instances of size $P$; instance $k$ trades only where its own limit $b + k\,d$ crosses, and the sweep stops at the first instance that cannot trade; every trade priced at the resting entry; one row per (resting entry, incoming instance); the remainder resting as instance $k_e = \lfloor M/P \rfloor$ at $b + k_e\,d$, showing $v_1 = \min(P - u, R)$, with no new timestamp, its later slices continuing the ladder; never IOC. Hence W's incoming-iceberg branch (findings F5, F20, F21). OQ-35 (an instance priced beyond §2.3) cannot arise: the precondition keeps every instance within §2.3 (section 4).
- **The FOK pre-check (§4.2.1 Rule 1, OQ-11).** Hidden quantity counts, ICEwPI slice by slice at its drifted price. Hence W's `matchable` and M4.
- **Eager activation and the shown slice (§4.2.1 Rule 4 *Slice activation timing*, OQ-20 correction note and its 2026-10-08 answer).** Hence finding F6: W is strict, and the book after every submission shows a slice for every iceberg with remaining quantity.
- **Ownership blindness (§6.6, review §2).** Hence M6.
- **Bounds (§2.3, §13.6 item 1).** Hence the dataset precondition and acceptance (F16).
- **Capacity (review §5).** Hence the five-ceiling precondition, the independent pressure axes (F2) and the reported discard rate.
- **The judging limit (review D3-08).** Hence W.

In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- §4.5: what each execution restriction does -----------------------------------------------------
    SpecQuote(
        "market-spec.md", "§4.5",
        "| NON | Yes | Yes | Regular, Block |",
        note="NON: partial fills allowed, the residual enters the book (W rests it).",
    ),
    SpecQuote(
        "market-spec.md", "§4.5",
        "| IOC | Yes | No (cancelled) | Regular |",
        note="IOC: partial fills allowed, the residual is cancelled (W: cause ioc_remainder).",
    ),
    SpecQuote(
        "market-spec.md", "§4.5",
        "| FOK | No (full or cancel) | No (cancelled) | Regular, Linked basket |",
        note="FOK: all or nothing (W: cause fok_kill; M4).",
    ),
    SpecQuote(
        "market-spec.md", "§4.5",
        "| ICE | Yes (peak slices) | Yes (hidden qty rests) | Iceberg only |",
        note="ICE: fills slice by slice, the hidden quantity rests (W: remaining = visible + hidden).",
    ),
    # --- §4.2: icebergs ------------------------------------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§4.2",
        "Only the peak size is shown in the order book.",
        note="An iceberg enters with its peak visible; the rest is hidden.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2",
        "When a slice is exhausted, a new slice of peak size is activated and receives a new timestamp.",
        note="W activates the next slice at once, as a new entry with a new sequence number taken at that event (F6).",
    ),
    SpecQuote(
        "market-spec.md", "§4.2",
        "**ICEwPI (Iceberg with Price Increment):** Each new slice is priced at the previous slice price ± a "
        "fixed delta. Buy icebergs: negative delta (price decreases per slice). Sell icebergs: positive delta "
        "(price increases per slice).",
        note="Slice k of an iceberg is priced base + k * ppd; every ladder gets worse slice by slice.",
    ),
    # --- §4.2.1: iceberg interaction rules -----------------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "A Fill-or-Kill order's pre-match liquidity check considers the total iceberg quantity for "
        "opposite-side icebergs at crossing prices.",
        note="Rule 1: W's matchable volume counts hidden quantity.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "For **ICEwPI icebergs (PPD != 0)**, the pre-check must compute how many slices would actually cross "
        "at their natural drifted prices and count only that volume as matchable.",
        note="Rule 1 for ICEwPI: W counts slice by slice at base + k * ppd.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "A single aggressive order (NON, IOC, or FOK) can fill multiple iceberg slices within one match cycle.",
        note="Rule 2: W keeps matching the same incoming order against each new slice.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "After the visible peak is consumed, the iceberg regenerates a new slice (at its natural drifted price "
        "for ICEwPI) and matching continues against the same aggressive order until either the aggressive "
        "order is fully filled, the iceberg is fully exhausted, or the regenerated slice's price no longer "
        "crosses the aggressive order's limit.",
        note="Rule 2's stopping conditions are W's.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "Subject to §6.1, each regenerated slice is ranked as a new entry at its own (drifted) price. The aggressor "
        "always trades the best-ranked counter bid in the book next.",
        note="W ranks every slice at activation and always trades the best-ranked entry next.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "**(1) Total quantity.** It aggresses with its **total** quantity, not only its visible peak. It keeps "
        "matching while it remains executable and has quantity left.",
        note="Incoming iceberg: W sweeps its whole quantity (finding F5).",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "Below, $b$ is the incoming iceberg's price, $P$ its peak (reduced part), $d$ its price increment (PPD; $d = 0$ "
        "for a standard iceberg), $M$ the quantity it sweeps on entry and $R$ its remainder.",
        note="The symbols of W's incoming-iceberg branch (finding F5).",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "**(2) One transaction per (counter bid, instance) pair.** The incoming iceberg is a chain of instances, each of "
        "size $P$; the last instance may be smaller. Each trade is between one resting counter-bid entry and one "
        "incoming instance.",
        note="Incoming iceberg: W walks it instance by instance, one row per (resting entry, incoming instance).",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "A trade row ends where a resting entry ends **or** where one incoming instance ends and the next begins. Two "
        "resting instances are never merged into one row.",
        note="W's row quantity: min(what the incoming order has left, the resting entry's visible, the rest of the "
             "incoming instance).",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "**(3) Price.** Every trade prices at the resting entry's limit (§6.2 item 3). For a resting ICEwPI that is the "
        "matched slice's drifted price $\\mathit{base} + k\\,\\mathit{ppd}$ (§5.1.2; OQ-31).",
        note="Every W row of an incoming iceberg prints at the resting entry's price, never at the instance limit.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "If a remainder $R > 0$ is left once the order is no longer executable, it enters the book **as an iceberg** "
        "with total $R$. With $k_e = \\lfloor M/P \\rfloor$ and $u = M - k_e P$ (the part of instance $k_e$ already "
        "traded), the remainder keeps its place in the ladder: it continues as instance $k_e$, at limit $b + k_e\\,d$.",
        note="W rests the remainder at b + k_e d as instance k_e of its ladder (asserted by the book check, F6).",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "It shows $v_1 = \\min(P - u, R)$, the unused part of instance $k_e$, and keeps $R - v_1$ hidden.",
        note="W's rested display: visible v1, hidden R - v1 (book check, F6; remaining quantity).",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "As for a NON residual, partial execution assigns no new timestamp (§5.2). The display after a part-used "
        "instance ($u > 0$) is derived from OMIE data, not stated by the sources (OQ-34 resolution).",
        note="The rested remainder keeps its arrival number in W; the v1 display at u > 0 is data-derived (F5).",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "**(5) Never IOC.** An iceberg is never an IOC order (§4.2). The remainder is not cancelled, and no output may "
        "label it IOC.",
        note="W cancels nothing of an incoming iceberg; any cancellation on one is a W mismatch.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "Instance $k$ ($k = 0, 1, 2, \\dots$) trades only at prices its own limit $b + k\\,d$ crosses; for $d = 0$ every "
        "instance has limit $b$. The sweep stops at the first instance that cannot trade, even if $b$ would still cross.",
        note="W's sweep limit for instance k = floor(M / P) is b + k d, and its stopping rule (finding F5).",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "*Simulator limit:* on a trade ring without room for the split rows of item (2), the rows are left unsplit "
        "(volumes and prices unchanged; a certificate gap, not a market rule).",
        note="Not reached on a valid draw: rows are bounded by trading units, kept at 80 below the ring's 100 (F5).",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "The hop trades at the **iceberg slice's own limit price**: the base price for PPD = 0, or the slice's "
        "drifted price $\\mathit{base} + k\\,\\mathit{ppd}$ for ICEwPI.",
        note="Rule 3 (OQ-31): W prices every slice trade at the slice's own price, hop included.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "**the entire cascade MUST resolve to a fixed point within the originating matching process** (all "
        "resultant trades carry that matching process's timestamp; the order book is never left in a crossed "
        "state between aggressor arrivals)",
        note="Rule 4: every slice trade belongs to the submission that triggered it (W's submitting-order id).",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "Only the first alternative has source support. Activation is **eager**.",
        note="Finding F6: W activates a slice inside the matching process that used up the previous one.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "So in the market, between matching processes, an iceberg with remaining quantity always shows a slice.",
        note="Finding F6: the book check after every submission (slice shown).",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "The incoming iceberg aggresses with its total quantity as a chain of instances: instance $k$ trades only where "
        "its own limit $b + k\\,d$ crosses (Rule 2, *Incoming iceberg*, items (2) and (6)). Only its non-executable "
        "remainder rests, and only after the sweep ends, at the rest price $b + k_e\\,d$ with $k_e = \\lfloor M/P \\rfloor$ "
        "(item (4))",
        note="Rule 3, ICE aggressor: W rests the remainder only after its sweep, at b + k_e d.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "On every trade of the sweep, the iceberg that was **already resting** is the price-setter. That includes slices "
        "regenerated during the sweep. The trade prints at that slice's own limit: the base price, or "
        "$\\mathit{base} + k\\,\\mathit{ppd}$ for ICEwPI. It never prints at the incoming iceberg's limit.",
        note="Rule 3, ICE aggressor: every W row of the sweep prints at the resting slice's own price.",
    ),
    # --- §5.2: timestamps ----------------------------------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§5.2",
        "A **new timestamp** is assigned on: modification (price, quantity, execution restriction), "
        "reactivation from inactive, or activation of a new iceberg slice.",
        note="A new slice queues behind every order already at its price.",
    ),
    SpecQuote(
        "market-spec.md", "§5.2",
        "**Partial execution is NOT a modification and does NOT assign a new timestamp.**",
        note="A partly filled order keeps its place; a NON residual rests with its entry timestamp.",
    ),
    SpecQuote(
        "market-spec.md", "§5.2",
        "All timestamps across orders and explicit capacity requests are assigned from a single global queue "
        "(unique, sequential).",
        note="W's single global sequence counter.",
    ),
    # --- §6.1 / §6.2: priority and matching ------------------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§6.1",
        "**Buy side:** ranked highest price first; ties broken by earliest timestamp.",
        note="A SELL incoming order meets bids highest first.",
    ),
    SpecQuote(
        "market-spec.md", "§6.1",
        "**Sell side:** ranked lowest price first; ties broken by earliest timestamp.",
        note="A BUY incoming order meets asks lowest first.",
    ),
    SpecQuote(
        "market-spec.md", "§6.1",
        "A match occurs when best bid price ≥ best ask price.",
        note="W's book is never left crossed after a submission (checked inside W as a self-consistency guard).",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 1",
        "Incoming order attempts to match against best counterpart in the consolidated order book.",
        note="W: the incoming order is the only new-timestamp order of a submission.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 2",
        "Match is possible if: orders are opposite sides, same contract, and `sell_price ≤ buy_price`.",
        note="W's crossing test, in integer ticks.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 3",
        "Trade executes at the **resting order's limit price** (maker pricing).",
        note="Every W trade prints at the resting entry's price.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 4",
        "If incoming order is partially filled and has NON restriction: residual enters the book with its "
        "original entry timestamp (see §5.2).",
        note="W rests a NON (or ICE) residual with the sequence number it entered with.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 5",
        "If incoming order is partially filled and has IOC restriction: residual is cancelled.",
        note="W cancels the IOC residual (cause ioc_remainder); M3.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 6",
        "If matching exhausts one price level, continues to next best price level.",
        note="W's multi-level walk.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 7",
        "Iceberg: when a single incoming order matches multiple slices of the same iceberg, the iceberg is "
        "always the price-setter.",
        note="Every slice trade prints at the slice's price.",
    ),
    # --- §6.6: no self-trade prevention -------------------------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§6.6",
        "The XBID/M7 matching engine does **not** implement self-trade prevention.",
        note="M6 and W are owner-blind: owners never enter the walk.",
    ),
    SpecQuote(
        "market-spec.md", "§6.6",
        "**Simulator implication:** The matching engine must not reject or skip matches based on order "
        "ownership.",
        note="M6: relabelling owners leaves the trade list unchanged, self-matches included.",
    ),
    # --- bounds ---------------------------------------------------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§2.3",
        "**Quantity bounds:** 0.1 MW (minimum) to 999 MW (maximum).",
        note="Every drawn quantity stays within 1 to 9 990 ticks (precondition).",
    ),
    SpecQuote(
        "market-spec.md", "§2.3",
        "**Price bounds:** −9,999 EUR/MWh (minimum) to +9,999 EUR/MWh (maximum).",
        note="Every price, and every ICEwPI slice price, stays within 0 to 1 999 800 ticks (precondition).",
    ),
    SpecQuote(
        "market-spec.md", "§13.6 item 1",
        "**Sign rule (hard).** ICEwPI buy orders must have `PPD < 0`; ICEwPI sell orders must have `PPD > 0`.",
        note="Precondition; it is also what makes every slice ladder monotone (findings F6, F7).",
    ),
    # --- open questions -------------------------------------------------------------------------------------------
    SpecQuote(
        "open-questions.md", "OQ-11",
        "**Resolution:** **Resolved. Interpretation B is correct.**",
        block="**Resolution:**",
        note="The FOK pre-check counts ICEwPI slices individually at their drifted prices; no price capping.",
    ),
    SpecQuote(
        "open-questions.md", "OQ-20",
        "Between matching processes, an iceberg with remaining quantity always shows a slice in the real order "
        "book.",
        block="**CORRECTION NOTE (2026-10-07",
        note="Finding F6: the lazy tolerance of r1/r2 is withdrawn.",
    ),
    SpecQuote(
        "open-questions.md", "OQ-20",
        '"No visible row" between matching processes is not a market outcome',
        block="**ANSWER TO CORRECTION-NOTE ITEM (d)",
        note="Finding F6: 'slice shown' is checked here.",
    ),
    SpecQuote(
        "open-questions.md", "OQ-32",
        "A slice activated within a matching process takes its queue place at that moment: after the order whose "
        "matching process activated it, behind every order then resting at its price, and ahead of every order "
        "that arrives later",
        block="**SCOPE CORRECTION (2026-10-08",
        note="E2: W numbers a slice at its activation event (queued for the next spec revision; v2.14 §5.2 already "
             "states the rules it rests on).",
    ),
    SpecQuote(
        "open-questions.md", "OQ-33",
        "Each resting NON order is one counter bid, each instance of a resting iceberg is its own counter bid, and two "
        "resting instances are never merged into one row. This holds for standard icebergs ($d = 0$) and for ICEwPI.",
        block="**Revised by user ruling (2026-10-20",
        note="W's incoming-iceberg rows split at resting entries and at incoming instances, for every d.",
    ),
    SpecQuote(
        "open-questions.md", "OQ-33",
        "Its other half stands: no row spans two resting entries.",
        block="**Revised by user ruling (2026-10-20",
        note="No W row ever spans two resting entries.",
    ),
    SpecQuote(
        "open-questions.md", "OQ-34",
        "Instance $k$ trades only at prices its own limit $b + k\\,d$ crosses. The sweep stops at the first instance "
        "that cannot trade, even if $b$ would still cross. Trades stay priced by the resting entry (OQ-31).",
        block="**Resolved by user ruling (2026-10-20",
        note="W's per-instance limits and stop rule: a rule now, no longer a modelling choice (finding F18 retired).",
    ),
    SpecQuote(
        "open-questions.md", "OQ-34",
        "A remainder $R > 0$ keeps its place in the ladder: it rests as instance $k_e = \\lfloor M/P \\rfloor$ at "
        "$b + k_e\\,d$, and later instances continue from there. The drift does not restart at slice 0.",
        block="**Resolved by user ruling (2026-10-20",
        note="W's rested remainder continues the ladder: b + (k_e + 1) d, ... (finding F21 counts it).",
    ),
    SpecQuote(
        "open-questions.md", "OQ-34",
        "With $u = M - k_e P$, the rested instance shows $v_1 = \\min(P - u, R)$, its unused part, and $R - v_1$ is "
        "hidden.",
        block="**Resolved by user ruling (2026-10-20",
        note="W's rested display v1 (book check, F6).",
    ),
    SpecQuote(
        "open-questions.md", "OQ-35",
        "An ICEwPI's instance $k$ is priced $b + k\\,d$ (Annex §4.5). For a SELL with $d > 0$ or a BUY with $d < 0$, a "
        "deep enough instance can fall beyond the SIDC limits of ±9,999 EUR/MWh.",
        block="**The question.",
        note="Open, and not reached: the precondition keeps every instance price within §2.3 (section 4).",
    ),
    # --- the domain review ------------------------------------------------------------------------------------------
    SpecQuote(
        "scenario-ladder-review.md", "§2",
        "**Moved into scope** as metamorphic relation **M6** (agent-id permutation leaves the trade list "
        "unchanged, including a self-match), asserted in C3 and swept at scale in C15.",
        note="M6 at scale, with a self-match asserted present (F9).",
    ),
    SpecQuote(
        "scenario-ladder-review.md", "§5 item 2",
        "A multi-level walk, an iceberg cascade, a settlement draw or a fuzz session can each exceed 100 trades, "
        "and a wrap silently corrupts I1/I2/I6 — *the very invariants used as the oracle*.",
        note="Why the trade-ceiling margin (80) is part of the precondition, and checked before the oracle.",
    ),
    SpecQuote(
        "scenario-ladder-review.md", "§5 item 3",
        "Also: `book_depth` and `max_icebergs` are independent axes (per `P3-PERF-REGEN-CAP-1`) and the fuzz "
        "grid must bound them independently.",
        note="Finding F2: the two pressure axes, swept and bounded independently.",
    ),
    SpecQuote(
        "scenario-ladder-review.md", "§5",
        "Added requirement: the precondition is checked **per draw and the discard rate reported**.",
        note="Discards are counted per variation and per pressure level.",
    ),
    SpecQuote(
        "scenario-ladder-review.md", "D3-08",
        "The fix belongs at a scenario oracle, not in I4.",
        note="Finding F5: W is that scenario oracle.",
    ),
    SpecQuote(
        "scenario-ladder-review.md", "D3-08",
        "The battery trusts the engine's queue for entries it does not hold in the pre-snapshot.",
        note="The judging limit W closes: regenerated and hidden slices.",
    ),
)

CITATIONS = (
    Citation(
        "Continuous Trading Matching Algorithm (public description) §5 — docs/omiecid/corpus/"
        "Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/"
        "Continuous_Trading_Matching_Algorithm_Public_Description.md:454-457",
        "Matching of orders is based on their ranking, which follows the  price-time-priority principle "
        "(cf. section 4.4). Orders are always executed at the best possible price, meaning that the best "
        "(i.e. highest-price) buy order is always matched with the best (i.e. lowest-price) sell order first. "
        "Orders with the same price limit are prioritized by their time stamp oldest first.",
    ),
    Citation(
        "Continuous Trading Matching Algorithm (public description) §5.1, §5.1.1, §5.1.2 — same file:462-479",
        "Regular matching is triggered by the entry of an order with a new time stamp. An order with a new "
        "time stamp may be a newly entered order, a modified order, a (re)activated order that was inactive "
        "before, or a new slice of an iceberg order. 5.1.1. Price determination When two orders are matched in "
        "a regular match, one of these orders must be an order with a new timestamp and the other one must be "
        "an order already present in the order book. The price at which two orders are matched becomes the "
        "price of the trade that is concluded. Two orders are matched at the limit price of the order that was "
        "already in the order book. If a buy order with a new timestamp is matched against an existing sell "
        "order, the limit price of the sell order becomes the trade execution price. If a sell order with a "
        "new timestamp is matched against an existing buy order, the limit price of the buy ord er becomes "
        "the trade execution price. 5.1.2. Iceberg orders in regular matching In a matching process where a "
        "single order with a new timestamp is matched against more than one slice of an iceberg order already "
        "in the order book, the price is always determined by the iceberg order already in the order book and "
        "never by the order with the new timestamp, even if the timestamp of the iceberg order is renewed "
        "during the matching process.",
    ),
    Citation(
        "Continuous Trading Matching Algorithm (public description) §5.1.3 — same file:484-493",
        "5.1.3. Matching against multiple orders If an order with a new timestamp can be executed, it is not "
        "necessarily executed at a single price (except orders with the execution restriction AON ), but may "
        "sequentially generate multiple transactions at different prices against multiple different orders "
        "that already existed in the order book. As soon as the order has been executed against all orders at "
        "a certain price limit, the next best price level becomes best and the order continuous to be matched "
        "against orders entered at this price level. This process continuous as long as the incoming order "
        "remains executable and has a positive order quantity. Subsequently , the order is deleted if the "
        "order quantity has reached zero or if it has the IOC execution restriction. In all other cases, the "
        "order is entered into the order book with its remaining quantity.",
    ),
    Citation(
        "docs/omiecid/scenario-ladder.md §C15, \"How it is certified\" and \"Why a shrinker is not optional\" "
        "(not readable by the quote checker)",
        "Each session runs through the engine twice (eager and jitted, asserted identical), and then every "
        "invariant I1–I9 and every metamorphic relation M1–M6 is evaluated on it. The pass criterion is **zero "
        "exceptions across every draw**: one violation is a certification failure, not a statistic. […] The "
        "shrinker takes the failing seed and bisects — repeatedly truncating the tape and re-running — until "
        "it finds the shortest prefix that still violates the same invariant.",
    ),
    Citation(
        "docs/omiecid/scenario-ladder-plan.md §The capacity-validity precondition (five ceilings)",
        "The simulator has five hard ceilings, all consequences of fixed-size arrays and fixed loop bounds: "
        "100 resting orders per side, 100 trades per contract in the trade tape, 200 matching iterations, "
        "1024 concurrent icebergs, and 64 slices per iceberg (`MAX_SLICES_PER_REGEN`, `jax_constants.py` §B2).",
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))
print(f"{sum(c.ok for c in QUOTE_CHECKS)} of {len(QUOTE_CHECKS)} quotes match their source")

## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`VARIATIONS = "ab"`**: the variations run, in the card's order. `a` = **C15a** smoke (200 draws), `b` = **C15b** gate (2 000 draws, the merge-gate figure), `c` = **C15c** deep (50 000 draws, nightly, not in the gate: add `c` to run it). Requested draws = the sum over the variations run.
- **Seeds.** Variation $v$ draws its $j$-th slot from seed `SEED0 + SEED_BASE[v] + j`. A slot whose draw is discarded (generation breach, I8 invalid, a failed precondition or a run that raised) is re-drawn **in the same cell** from replacement seeds `SEED0 + SEED_BASE[v] + DRAWS_v + j·REPLACEMENTS_PER_SLOT + r − 1`, $r = 1 \ldots$ `REPLACEMENTS_PER_SLOT`, so every cell keeps its share. The generator's own internal redraw streams (`seed + attempt · 1 000 003`, at most 24 attempts) stay inside each variation's range, since the ranges are $10^8$ apart.
- **The grid** (section 4): mix × crossing probability (0.05, 0.2, 0.4, 0.6) × tie density (0, 0.4) × length (32, 96, 160) × book depth (100, 60) × iceberg slots (1 024, 64), the last two being the pressure axes of finding F2.
- **`EAGER_PER_MIX = 1`**, **`EAGER_LENGTH = 32`**: eager comparisons per order-type mix, on its first valid draws of that length (finding F13).
- **`RELATIONS_PER_DRAW = 1`**: relations from the rotation M1, M2, M3, M5, M6 applied per draw (finding F10); M4 runs on every FOK regardless. **`MIN_RELATION_APPLICATIONS = 5`**: the coverage each relation must reach per variation where its mix allows it.
- **`SHRINK_MAX_DRAWS = 3`**, **`SHRINK_MAX_EVALS = 150`**: failing draws shrunk (in order of failure), and the evaluation budget per shrink (finding F14). A failing draw beyond the first three is frozen unshrunk.
- **The five ceilings** are read from the configuration dataclasses and `jax_constants.py`, never typed in; section 7 asserts they are the documented values (100, 100, 200, 1 024, 64).

In [ ]:
CERT_ID = "C15"                 # C00-C16 are the ladder; C90-C99 are reserved for the harness
VARIATIONS = "ab"               # a = C15a smoke, b = C15b gate; add "c" for C15c deep (nightly)
DRAWS_A = 200                   # C15a: NON and IOC only, short sessions
DRAWS_B = 2000                  # C15b: full grid, all order types (the merge-gate figure)
DRAWS_C = 50000                 # C15c: full grid weighted toward iceberg and FOK interaction
SEED0 = 0                       # variation v, slot j: seed SEED0 + SEED_BASE[v] + j
REPLACEMENTS_PER_SLOT = 4       # re-draws per slot (same cell) after a discard
EAGER_PER_MIX = 1               # eager comparisons per order-type mix (finding F13)
EAGER_LENGTH = 32               # ... on draws of this session length
RELATIONS_PER_DRAW = 1          # relations from the rotation M1, M2, M3, M5, M6 per draw (F10)
MIN_RELATION_APPLICATIONS = 5   # per variation and relation, where the mix allows it
SHRINK_MAX_DRAWS = 3            # failing draws shrunk (finding F14)
SHRINK_MAX_EVALS = 150          # evaluation budget per shrink

In [ ]:
import json
import math
import os
import pathlib
from collections import namedtuple

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.encoding import decode_price, decode_qty, encode_price, encode_qty
from omiecid_simulator.omiecid.jax_constants import MAX_SLICES_PER_REGEN
from omiecid_simulator.omiecid.omie_config import OMIEMarketConfig, OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side, Validity

CONSTRUCTION = "C15.random_session"   # the generator's fuzz mode (make_random_session over fuzz_grid regimes)
CONFIG = OMIEProductConfig()          # builds tapes and decodes for display; never a source of the expectation
DEFAULT_MARKET = OMIEMarketConfig()

BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, IOC, FOK, ICE = (int(ExecRestriction.NON), int(ExecRestriction.IOC), int(ExecRestriction.FOK),
                      int(ExecRestriction.ICE))
TYPE_NAME = {NON: "NON", IOC: "IOC", FOK: "FOK", ICE: "ICE"}
GFS = int(Validity.GFS)
C_CONTRACT, C_SIDE, C_PRICE, C_QTY, C_OID, C_AGENT, C_TS = (synth.COL_CONTRACT, synth.COL_SIDE, synth.COL_PRICE,
                                                             synth.COL_QTY, synth.COL_OID, synth.COL_AGENT,
                                                             synth.COL_TS)
C_EXEC, C_VALIDITY, C_EXPIRY, C_PEAK, C_PPD = (synth.COL_EXEC, synth.COL_VALIDITY, synth.COL_EXPIRY,
                                               synth.COL_PEAK, synth.COL_PPD)
PRICE_INT_MAX = encode_price(CONFIG.price_max, CONFIG)   # 1 999 800 (§2.3), in ticks of 0.01 EUR/MWh
QTY_INT_MIN = encode_qty(CONFIG.qty_min, CONFIG)         # 1 (§2.3), in ticks of 0.1 MW
QTY_INT_MAX = encode_qty(CONFIG.qty_max, CONFIG)         # 9 990 (§2.3)

# ---- the five ceilings, read from the configuration (never typed in) -----------------------------------------
CEILINGS = {
    "book_depth (resting orders per side)": int(DEFAULT_MARKET.book_depth),
    "nTrades (trade rows per contract)": int(DEFAULT_MARKET.nTrades),
    "max_match_iterations (per submission)": int(DEFAULT_MARKET.max_match_iterations),
    "max_icebergs (concurrent icebergs)": int(DEFAULT_MARKET.max_icebergs),
    "MAX_SLICES_PER_REGEN (slices per iceberg)": int(MAX_SLICES_PER_REGEN),
}
DOCUMENTED_CEILINGS = [100, 100, 200, 1024, 64]   # the capacity-validity precondition's five values
SLICE_MARGIN = 64                                 # generator margin: ceil(total / peak) <= 64

# ---- the pressure axes (finding F2): configured ceilings, each with an 80 % generation margin -----------------
DEPTH_LEVELS = (int(DEFAULT_MARKET.book_depth), 60)       # book depth D (resting orders per side)
SLOT_LEVELS = (int(DEFAULT_MARKET.max_icebergs), 64)      # iceberg slots M (concurrent icebergs)


def resting_margin(depth):
    return synth.DEFAULT_ENVELOPE.max_resting_per_side if depth == DEPTH_LEVELS[0] else (4 * depth) // 5


def slot_margin(slots):
    return synth.DEFAULT_ENVELOPE.max_concurrent_icebergs if slots == SLOT_LEVELS[0] else (4 * slots) // 5


PRESSURES = tuple((d, m) for d in DEPTH_LEVELS for m in SLOT_LEVELS)
MARKET_OF = {(d, m): OMIEMarketConfig(book_depth=d, max_icebergs=m) for (d, m) in PRESSURES}
ENVELOPE_OF = {(d, m): synth.CapacityEnvelope(max_resting_per_side=resting_margin(d),
                                              max_trades=synth.DEFAULT_ENVELOPE.max_trades,
                                              max_hops=synth.DEFAULT_ENVELOPE.max_hops,
                                              max_concurrent_icebergs=slot_margin(m))
               for (d, m) in PRESSURES}

# ---- the regime grid ------------------------------------------------------------------------------------------------
MIXES = tuple(synth.FUZZ_TYPE_MIXES)          # non, non_ioc, non_ioc_fok, non_ioc_fok_ice, all_icewpi
CROSS_PROBS = (0.05, 0.2, 0.4, 0.6)
TIE_DENSITIES = (0.0, 0.4)
LENGTHS = (32, 96, 160)
Cell = namedtuple("Cell", "mix cross tie n depth slots")


def regime_of(cell):
    return synth.fuzz_grid(mixes=(cell.mix,), cross_probs=(cell.cross,), tie_densities=(cell.tie,),
                           lengths=(cell.n,))[0]


def feasible(cell):
    # structural: a tape carries no cancellations, so n orders need two sides of the resting margin (F2)
    return cell.n <= 2 * resting_margin(cell.depth)


def cells_for(mixes, lengths):
    return [c for c in (Cell(mx, cp, td, n, d, m) for mx in mixes for cp in CROSS_PROBS for td in TIE_DENSITIES
                        for n in lengths for (d, m) in PRESSURES) if feasible(c)]


def c15c_weight(cell):
    types_ = set(int(t) for t in synth.FUZZ_TYPE_MIXES[cell.mix])
    return 3 if {FOK, ICE} <= types_ else 2 if FOK in types_ else 1


CELLS = {
    "a": cells_for(("non", "non_ioc"), (32,)),
    "b": cells_for(MIXES, LENGTHS),
}
CELLS["c"] = list(CELLS["b"])
SWEEP = {
    "a": list(CELLS["a"]),
    "b": list(CELLS["b"]),
    "c": [c for c in CELLS["c"] for _ in range(c15c_weight(c))],
}
REGIME = {c: regime_of(c) for c in CELLS["b"]}
DRAWS_OF = {"a": int(DRAWS_A), "b": int(DRAWS_B), "c": int(DRAWS_C)}
SEED_BASE = {"a": 0, "b": 100_000_000, "c": 200_000_000}
LABEL = {"a": "C15a", "b": "C15b", "c": "C15c"}
RUN_VARIATIONS = [v for v in "abc" if v in str(VARIATIONS)]
REQUESTED = sum(DRAWS_OF[v] for v in RUN_VARIATIONS)


def slot_seed(v, j, r=0):
    base = int(SEED0) + SEED_BASE[v]
    return base + j if r == 0 else base + DRAWS_OF[v] + j * int(REPLACEMENTS_PER_SLOT) + (r - 1)


_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C15_randomised_fuzz.ipynb"
REPO_ROOT = pathlib.Path(synth.__file__).resolve().parents[2]
SHRUNK_DIR = pathlib.Path(os.environ.get("OMIE_CERT_FIXTURE_DIR") or (REPO_ROOT / "tests" / "omiecid" / "fixtures" / "cert"))

print(f"{CERT_ID}: {CONSTRUCTION}; variations {[LABEL[v] for v in RUN_VARIATIONS]}; requested draws {REQUESTED}")
for v in "abc":
    print(f"  {LABEL[v]}: {DRAWS_OF[v]} draws over {len(CELLS[v])} cells (sweep length {len(SWEEP[v])}), "
          f"seeds from {slot_seed(v, 0)}{'' if v in RUN_VARIATIONS else '  [not run]'}")
print("the five ceilings (configuration):")
for _k, _v in CEILINGS.items():
    print(f"  {_k}: {_v}")
print("pressure levels (book depth D, iceberg slots M) -> generation envelope:")
for _p in PRESSURES:
    print(f"  D={_p[0]:4d} M={_p[1]:5d}: {ENVELOPE_OF[_p]}")
print(f"slice margin <= {SLICE_MARGIN}; eager: {EAGER_PER_MIX} per mix at n = {EAGER_LENGTH}; "
      f"relations per draw: {RELATIONS_PER_DRAW}; shrink: {SHRINK_MAX_DRAWS} draws, {SHRINK_MAX_EVALS} evaluations")
print(f"shrunk fixtures -> {SHRUNK_DIR}")

## 4. Dataset

One draw is a whole session from one integer seed, in the card's four steps, by the generator's public API (`omie_tape_synth.make_random_session` over `fuzz_grid` regimes):

1. **The regime cell.** Slot $j$ of variation $v$ takes cell `SWEEP[v][j mod len(SWEEP[v])]`: the grid is swept systematically, never sampled, so every cell gets its share. A cell is an order-type mix (`non`; `non_ioc`; `non_ioc_fok`; `non_ioc_fok_ice`; `all_icewpi`, which adds a price increment to about half the icebergs), a crossing probability (how often an order is priced through the session's mid), a tie density (the probability an order reuses the previous tape timestamp), a session length (32, 96 or 160 orders) and the two pressure axes (book depth $D$, iceberg slots $M$: finding F2). Each pressure level is a configured engine ceiling (`MARKET_OF`) with its own generation envelope (`ENVELOPE_OF`).
2. **The orders.** Side, distance from the mid, size, type, owner (1–4, with a small common-owner bias for M6) and, for icebergs, peak ($\lfloor q/k \rfloor$, $k \in [2,5]$, so at most 9 slices) and price increment (sign per §13.6) come from the regime's laws; arrival gaps are $1 + \lfloor \mathrm{Exp}(1) \rfloor$ ticks.
3. **The capacity bounds.** The generator rejection-samples against the cell's envelope on all four analytic axes independently and redraws on any breach (`params["retries"]`); the slice margin holds by the peak law and is re-checked below. A draw that cannot be made inside the envelope raises `CapacityBreach`, which section 7 records as a **discard** and replaces from the same cell.
4. **Emission.** The 13-column int32 tape, with the seed, the regime (`params`) and the content digest (`Draw.digest()`).

**The dataset precondition** (`precondition(draw, cell)` below; asserted on every draw in section 7, a failing draw is discarded and recorded as a failed check): real rows first then pad rows; exactly $n$ real rows; one contract id; distinct order ids $\ge 1$; tape timestamps non-decreasing; owners in 1–4; every row GFS with no expiry; types within the cell's mix; prices within 0–1 999 800 ticks and quantities within 1–9 990 ticks (§2.3); a non-iceberg row carries no peak and no increment; an iceberg row has $0 < \mathit{peak} \le \mathit{quantity}$, $\lceil \mathit{quantity}/\mathit{peak} \rceil \le 64$, an increment only in the `all_icewpi` mix and then with the §13.6 sign, and every slice price within §2.3 (the same bound covers an incoming iceberg's instance chain: its instance limits $b + k\,d$ and a rested remainder's price $b + k_e\,d$ lie on the one ladder $k \le \lceil q/P \rceil - 1$, so OQ-35, an instance beyond ±9,999 EUR/MWh, cannot arise on a valid draw); the draw's analytic bounds equal `tape_capacity_bounds(tape)` and lie inside the cell's envelope on all four axes.

Below: the cell counts per variation, then one sample draw per mix in readable units beside the raw integers, and the shape of one session.

In [ ]:
from collections import Counter

from IPython.display import Markdown, display


def real_rows(tape):
    tape = np.asarray(tape).reshape(-1, synth.N_COLS)
    return tape[tape[:, C_OID] >= 1]


def ceil_div(a, b):
    return -(-int(a) // int(b))


def n_slices(row):
    return ceil_div(row[C_QTY], row[C_PEAK]) if int(row[C_EXEC]) == ICE and int(row[C_PEAK]) > 0 else 1


def make_draw(cell, seed):
    # One session of `cell` from `seed` (the generator's fuzz mode, the cell's envelope). May raise CapacityBreach.
    return synth.make_random_session(int(seed), REGIME[cell], CONFIG, envelope=ENVELOPE_OF[(cell.depth, cell.slots)])


def precondition(draw, cell):
    # The dataset precondition (section 4 text). Returns the list of problems (empty = holds).
    bad = []
    tape = np.asarray(draw.tape)
    if tape.dtype != np.int32 or tape.ndim != 2 or tape.shape[1] != synth.N_COLS:
        return [f"tape is not (N, 13) int32: {tape.dtype} {tape.shape}"]
    real = tape[:, C_OID] >= 1
    k = int(real.sum())
    if not real[:k].all() or real[k:].any():
        bad.append("real rows are not all before the pad rows")
    rows = tape[:k]
    if k != cell.n:
        bad.append(f"{k} real rows != session length {cell.n}")
    if len(set(int(x) for x in rows[:, C_CONTRACT])) > 1:
        bad.append("more than one contract id")
    oids = [int(x) for x in rows[:, C_OID]]
    if len(set(oids)) != len(oids):
        bad.append("duplicate order ids")
    if np.any(np.diff(rows[:, C_TS].astype(np.int64)) < 0):
        bad.append("tape timestamps decrease")
    allowed = set(int(t) for t in synth.FUZZ_TYPE_MIXES[cell.mix])
    for r in rows:
        oid, side, ex = int(r[C_OID]), int(r[C_SIDE]), int(r[C_EXEC])
        p, q, pk, ppd = int(r[C_PRICE]), int(r[C_QTY]), int(r[C_PEAK]), int(r[C_PPD])
        if side not in (BUY, SELL):
            bad.append(f"oid {oid}: side {side}")
        if ex not in allowed:
            bad.append(f"oid {oid}: restriction {ex} outside mix {cell.mix}")
        if int(r[C_AGENT]) not in (1, 2, 3, 4):
            bad.append(f"oid {oid}: owner {int(r[C_AGENT])}")
        if int(r[C_VALIDITY]) != GFS or int(r[C_EXPIRY]) != 0:
            bad.append(f"oid {oid}: not GFS without expiry")
        if not 0 <= p <= PRICE_INT_MAX:
            bad.append(f"oid {oid}: price {p} outside §2.3")
        if not QTY_INT_MIN <= q <= QTY_INT_MAX:
            bad.append(f"oid {oid}: quantity {q} outside §2.3")
        if ex != ICE:
            if pk != 0 or ppd != 0:
                bad.append(f"oid {oid}: non-iceberg row with peak {pk} / increment {ppd}")
            continue
        if not 0 < pk <= q:
            bad.append(f"oid {oid}: iceberg peak {pk} not in (0, {q}]")
            continue
        ns = ceil_div(q, pk)
        if ns > SLICE_MARGIN:
            bad.append(f"oid {oid}: {ns} slices > slice margin {SLICE_MARGIN}")
        if ppd != 0:
            if cell.mix != "all_icewpi":
                bad.append(f"oid {oid}: price increment outside the ICEwPI mix")
            if (side == BUY and ppd >= 0) or (side == SELL and ppd <= 0):
                bad.append(f"oid {oid}: increment {ppd} breaks the §13.6 sign rule")
            last = p + (ns - 1) * ppd        # also bounds every incoming instance limit and rest price (F5; OQ-35)
            if not 0 <= last <= PRICE_INT_MAX:
                bad.append(f"oid {oid}: last slice price {last} outside §2.3")
    bounds = synth.tape_capacity_bounds(tape)
    if dict(draw.capacity_bounds) != bounds:
        bad.append(f"draw bounds {dict(draw.capacity_bounds)} != tape bounds {bounds}")
    try:
        synth.check_envelope(ENVELOPE_OF[(cell.depth, cell.slots)], **bounds)
    except synth.CapacityBreach as exc:
        bad.append(f"outside the cell envelope: {exc}")
    return bad


def max_slices(tape):
    rows = real_rows(tape)
    return max([n_slices(r) for r in rows if int(r[C_EXEC]) == ICE] or [0])


# ---- the cells -------------------------------------------------------------------------------------------------------
_rows = ["| variation | draws | cells | sweep | mixes | lengths | pressure levels (D, M) |", "|---|---|---|---|---|---|---|"]
for v in "abc":
    cs = CELLS[v]
    _rows.append(f"| {LABEL[v]}{'' if v in RUN_VARIATIONS else ' (not run)'} | {DRAWS_OF[v]} | {len(cs)} | {len(SWEEP[v])} | "
                 f"{', '.join(sorted(set(c.mix for c in cs)))} | {sorted(set(c.n for c in cs))} | "
                 f"{sorted(set((c.depth, c.slots) for c in cs))} |")
display(Markdown("\n".join(_rows)))
_by_len = Counter((c.n, c.depth) for c in CELLS["b"])
print("C15b cells per (length, depth):", dict(sorted(_by_len.items())))

# ---- one sample draw per mix (C15b's first 32-order cell of the mix at default pressure) ---------------------------
SAMPLES = {}
for mx in MIXES:
    cell = next(c for c in CELLS["b"] if c.mix == mx and c.n == 32 and c.cross == 0.4 and c.tie == 0.4
                and (c.depth, c.slots) == PRESSURES[0])
    SAMPLES[mx] = (cell, make_draw(cell, slot_seed("b", 0)))

_rows = ["| mix | seed | digest | rows | IOC | FOK | ICE | ICEwPI | tie groups | max slices | bounds (rest, trades, hops, icebergs) | precondition |",
         "|---|---|---|---|---|---|---|---|---|---|---|---|"]
for mx, (cell, d) in SAMPLES.items():
    rr = real_rows(d.tape)
    ex = [int(r[C_EXEC]) for r in rr]
    ts = [int(r[C_TS]) for r in rr]
    ties = sum(1 for g in Counter(ts).values() if g > 1)
    b = d.capacity_bounds
    _rows.append(f"| {mx} | {d.seed} | `{d.digest()[:12]}` | {len(rr)} | {ex.count(IOC)} | {ex.count(FOK)} | {ex.count(ICE)} | "
                 f"{sum(1 for r in rr if int(r[C_EXEC]) == ICE and int(r[C_PPD]) != 0)} | {ties} | {max_slices(d.tape)} | "
                 f"{b['resting_per_side']}, {b['trades']}, {b['hops']}, {b['concurrent_icebergs']} | "
                 f"{'holds' if not precondition(d, cell) else precondition(d, cell)} |")
display(Markdown("\n".join(_rows)))

# ---- the first 16 orders of the ICEwPI sample, readable beside raw -----------------------------------------------------
_cell, _d = SAMPLES["all_icewpi"]
_rows = ["| sub | oid | side | type | price (EUR/MWh) | qty (MW) | peak (MW) | ppd (EUR/MWh) | price_int | qty_int | peak_int | ppd_int | owner | ts |",
         "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
for s, r in enumerate(real_rows(_d.tape)[:16]):
    ex = int(r[C_EXEC])
    _rows.append(
        f"| {s} | {int(r[C_OID])} | {'BUY' if int(r[C_SIDE]) == BUY else 'SELL'} | "
        f"{TYPE_NAME[ex] + ('wPI' if ex == ICE and int(r[C_PPD]) else '')} | {decode_price(int(r[C_PRICE]), CONFIG):.2f} | "
        f"{decode_qty(int(r[C_QTY]), CONFIG):.1f} | {decode_qty(int(r[C_PEAK]), CONFIG):.1f} | {int(r[C_PPD]) * CONFIG.price_tick:+.2f} | "
        f"{int(r[C_PRICE])} | {int(r[C_QTY])} | {int(r[C_PEAK])} | {int(r[C_PPD])} | {int(r[C_AGENT])} | {int(r[C_TS])} |")
display(Markdown(f"**`all_icewpi` sample**, cell `{_cell}`, seed {_d.seed}, regime `{_d.params['regime']}`, "
                 f"safe from index {_d.params['safe_from_index']}, generator retries {_d.params['retries']}\n\n"
                 + "\n".join(_rows)))

In [ ]:
import matplotlib.pyplot as plt

# The shape of one session: every order's limit price against its arrival, by side and type (the book it builds).
_cell, _d = SAMPLES["all_icewpi"]
_rr = real_rows(_d.tape)
_mid = float(np.median(_rr[:, C_PRICE]))
fig, ax = plt.subplots(figsize=(8, 3.2))
_marker = {NON: "o", IOC: "x", FOK: "s", ICE: "^"}
for ex in (NON, IOC, FOK, ICE):
    for side, colour in ((BUY, "tab:green"), (SELL, "tab:red")):
        sel = [(s, r) for s, r in enumerate(_rr) if int(r[C_EXEC]) == ex and int(r[C_SIDE]) == side]
        if sel:
            ax.scatter([s for s, _ in sel], [(int(r[C_PRICE]) - _mid) * CONFIG.price_tick for _, r in sel],
                       s=[8 + 2 * int(r[C_QTY]) ** 0.5 for _, r in sel], marker=_marker[ex], color=colour,
                       label=f"{'BUY' if side == BUY else 'SELL'} {TYPE_NAME[ex]}")
ax.axhline(0, color="black", lw=0.6)
ax.set_xlabel("submission (arrival order)")
ax.set_ylabel("limit price - median (EUR/MWh)")
ax.set_title(f"{CONSTRUCTION}, {LABEL['b']} sample, mix all_icewpi, seed {_d.seed}: orders by arrival")
ax.legend(loc="best", fontsize=7, ncol=4)
plt.tight_layout()
plt.show()

## 5. Known result

No per-draw answer exists in closed form; the oracle is the battery, the reference walk and the relations, and **one exception on one draw is a certification failure**. In integer ticks throughout.

**(B) The battery** on every valid draw's jitted run and on every variant run: `run_battery(rec, expects_trade = |W(tape)| > 0, envelope = the cell's envelope)`, RB and I1–I9 with no check at `fail`. `not_applicable` is a legitimate outcome and never a failure (finding F11 for I9). I8 `invalid` on the draw's own run discards the draw (re-drawn in the same cell); on a variant run it skips the relation for that draw.

**(W) The reference walk.** The real rows are processed in submission order. Every entry gets the next number of one global sequence when it enters, and an iceberg slice gets a new one when it is activated (§5.2, *"unique, sequential"*). For an incoming order $o$ of side $\sigma$, limit $L$ and quantity $Q$:

1. **FOK pre-check** (§4.2.1 Rule 1, OQ-11): $\mathit{matchable} = \sum_{e\ \bar\sigma} \big[\text{visible}_e \cdot \mathbb 1\{\text{cross}(p_e, L)\}\big] + \sum_{\text{icebergs } e} \sum_{k > k_e} x_{e,k} \cdot \mathbb 1\{\text{cross}(\mathit{base}_e + k\,\mathit{ppd}_e,\ L)\}$, with $x_{e,k}$ the sizes of the slices still hidden ($\mathit{peak}$, …, the remainder). If $\mathit{matchable} < Q$: no trade, the book unchanged, $o$ cancelled for $Q$ (`fok_kill`).
2. **The walk** (§6.1, §6.2 items 1–3, 6, 7; §4.2.1 Rules 2–4): while $o$ has quantity left and the best opposite entry crosses $L$ ($\text{sell} \le \text{buy}$), trade $f = \min(\text{left}_o, \text{visible}_e)$ **at $p_e$**, $e$ being the best price, then lowest sequence number — one row per resting entry. A resting slice used up is replaced **at once** by the iceberg's next slice — $\min(\mathit{peak}, \text{hidden})$ at $\mathit{base} + k\,\mathit{ppd}$ — numbered **at that event** (E2: after $o$, behind every entry then at its price, ahead of every later arrival; §4.2.1 Rule 4 eager activation, OQ-32 (3b)); it is ranked as a new entry and $o$ keeps matching whatever is then best; an exhausted iceberg leaves the book. **An incoming iceberg** (§4.2.1 Rule 2 *Incoming iceberg*, amended 2026-10-20; OQ-33 revised, OQ-34 resolved) of base $b = L$, peak $P$ and increment $d$ is walked the same way with its **total** quantity $Q$, as a chain of instances of size $P$ (items (1), (2), (6); finding F5): with $M$ the quantity swept so far, the instance in use is $k = \lfloor M/P \rfloor$ and its limit $L_k = b + k\,d$; the best opposite entry trades only if it crosses $L_k$, for $f = \min(\text{left}_o,\ \text{visible}_e,\ (k+1)P - M)$ at $p_e$, so a row ends where the resting entry ends **or** where instance $k$ ends; the sweep stops at the first instance whose limit the best entry does not cross, even if $b$ still would. For $d = 0$ every $L_k = b$ and only the row split remains.
3. **The residual**: NON rests what is left with the sequence number it entered with (§6.2 item 4, §5.2); an incoming iceberg's remainder $R = Q - M$ rests as an iceberg that keeps its place in the ladder: instance $k_e = \lfloor M/P \rfloor$ at $b + k_e\,d$, showing $v_1 = \min(P - u, R)$ with $u = M - k_e P$, $R - v_1$ hidden, and the number it entered with (no new timestamp, §5.2); its next slices continue the ladder at $b + (k_e + 1)\,d$, $b + (k_e + 2)\,d$, …, each $\min(P, \text{hidden})$ (items (4), (6)); nothing of it is cancelled (item (5)), and it does not cross the book, since the best opposite entry did not cross $L_{k_e}$; IOC cancels it (`ioc_remainder`, §6.2 item 5); an accepted FOK has none.

W predicts each trade as $[\text{buyer id}, \text{seller id}, \text{price}, \text{qty}, \text{submitting order id}]$, each order's cancellations $[(\text{cause}, \text{qty})]$, each order's remaining visible + hidden quantity, and, after every submission, the book each order is shown in: $[\text{side}, \text{price}, \text{visible}, \text{hidden}]$. **Strict** (finding F6): no activation reading is left open; a slice takes the number it got at its activation event, and nothing is relaxed.

**What is asserted per draw (W):** every order accepted, none rejected (F16); the engine's trade rows (`rec.fills` in record order, the submission named by its order id) equal W's, row for row, in count and content; per order, the cancellations (cause and quantity) equal W's, and `still_resting + hidden_remaining` equals W's remaining; no order evicted; and, on the engine's post-submission snapshot after **every** submission (while the trade rows still agree), each order W holds is shown at W's price with W's visible quantity and holds W's hidden quantity, and no other order is shown or holds hidden quantity (`book_after_submission`), and **no iceberg with remaining quantity shows no slice** (`slice_shown`, finding F6; judged on the engine's book alone). W's own book is uncrossed after every submission and shows a slice for every order it holds (self-consistency guards on the judging code, reported as `reference_internal` if they ever break). W also counts, on its own walk, how often the amended incoming-iceberg branch is exercised (finding F21; asserted in section 7).

**(M) The relations**, each on the draw's tape or a variant of it (findings F7–F10); every variant run must also pass (B) and (W):

| relation | variant | asserted (battery check) |
|---|---|---|
| **M1** | the tape with every eligible tie group reversed (F7) | equal aggregate matched volume; no price-priority violation involving a tie member (`check_m1_tie_permutation`) |
| **M2** | prefix ending at the first NON/IOC aggressor that trades in W; its quantity doubled, or its limit tightened by $1 + (\text{seed} \bmod 3)$ ticks | size up never matches less; limit tightened never matches more (`check_m2_monotonicity`) |
| **M3** | the same prefix with that aggressor's restriction flipped NON ↔ IOC | identical trade lists; the IOC's own remainder never rests (`check_m3_ioc_non_equivalence`) |
| **M4** | none: every FOK of the draw's own run | fills exactly its quantity, or no fill and a byte-identical book (`check_m4_fok_dichotomy`) |
| **M5** | prefix ending at the first NON/IOC aggressor that trades in W and whose quantity $Q$ divides by $k \in \{2, 3\}$; it is replaced by $k$ consecutive orders of $Q/k$ with fresh ids | equal (counterparty, price) → quantity maps (`check_m5_split_aggressor`) |
| **M6** | the whole tape with owners relabelled $1 \to 2 \to 3 \to 4 \to 1$ | identical trade list, owners mapped (`check_m6_ownership_blindness`) |

**(E)** eager == jitted on every integer field (`compare_records`) for the eager draws of section 3.

**The pre-registered band on aggregate matched volume** (reported, **never** asserted; finding F12). Every order is priced through the mid with probability $c$ (crossing) or away from it; two orders priced away from the mid never cross, and orders after the generator's safe-mode switch never trade, so **every trade has at least one crossing-priced leg among the first $n_{\mathrm{eff}}$ orders** (`params["safe_from_index"]`, or $n$). Hence the session's matched volume $V$ (each trade counted once) satisfies $V \le S$, where $S$ is the crossing-priced quantity among those orders, and $E[S] = c\, n_{\mathrm{eff}}\, \mu_q$, $\operatorname{Var}[S] = n_{\mathrm{eff}} (c\, \mu_{q^2} - c^2 \mu_q^2)$, with $\mu_q, \mu_{q^2}$ the first two moments of the size law $q = \min(\max(\operatorname{round}(10 X), 1), 600)$, $X \sim \mathrm{LogNormal}(1.6, 0.7)$, computed exactly from the normal CDF. For a cell with $k$ valid draws, mean effective length $\bar n$ and mean volume $\bar V$, the band is

$$\Big[\max\big(0,\ \lambda\, c\, \bar n\, \mu_q - z\,\mathrm{sd}\big),\ \ c\, \bar n\, \mu_q + z\,\mathrm{sd}\Big],\qquad \mathrm{sd} = \sqrt{\bar n\,(c\,\mu_{q^2} - c^2\mu_q^2)/k},\qquad \lambda = 0.05,\ z = 3.$$

The upper edge is the sound expectation bound; the lower edge is deliberately weak (a matching engine that fills almost nothing when $c$ is large is flagged). The constants $\lambda$ and $z$ are fixed here, before any run, and are never widened.

The code below implements W, the relation variants and the band, as functions of the tape and `draw.params` only. Nothing here reads `draw.expected` or runs the engine.

In [ ]:
def canon(x):
    # Canonical compact JSON: how every recorded value is written (in full).
    return json.dumps(x, sort_keys=True, separators=(",", ":"))


def crosses_in(side_in, limit, resting_price):
    # §6.2 item 2 for an incoming order of side_in at `limit` against a resting entry at `resting_price`.
    return resting_price <= limit if side_in == BUY else resting_price >= limit


class Entry:
    __slots__ = ("side", "price", "vis", "hidden", "oid", "seq", "ice")

    def __init__(self, side, price, vis, hidden, oid, seq, ice):
        self.side, self.price, self.vis, self.hidden, self.oid, self.seq, self.ice = side, price, vis, hidden, oid, seq, ice


class RefWalk:
    """(W) The independent price-time reference walk over one tape's real rows.

    observed=None: W alone (used for I9, the relations and the band).
    observed=[[buyer, seller, price, qty, submitting oid], ...]: strict lock-step comparison with the engine's rows
    (finding F6: a slice is activated eagerly and queued at its activation event, E2); the first divergence is
    recorded in `mismatch` and the walk stops.
    after=f: f(oid, walk) is called at the end of every submission (the book check, "slice shown").
    An incoming iceberg is walked as a chain of instances (v2.14 §4.2.1 Rule 2 "Incoming iceberg", amended
    2026-10-20; finding F5); `cov` counts how often that branch is exercised (finding F21).
    """

    def __init__(self, tape, observed=None, after=None):
        self.rows = real_rows(tape)
        self.obs = observed
        self.after = after
        self.k = 0
        self.book = []
        self.seq = 0
        self.trades = []
        self.cancel = {}
        self.mismatch = None
        self.internal = []
        self.activations = 0
        self.rested = {}                             # oid -> (k_e, d) of every rested incoming-iceberg remainder
        self.cov = Counter()                         # coverage of the incoming-iceberg branch (reported; F21)

    def nxt(self):
        self.seq += 1
        return self.seq

    def activate(self, e):
        # A resting iceberg's slice is used up: its next slice is made available at once (§4.2; §4.2.1 Rule 4,
        # "Activation is eager") and takes its queue place at this event (E2; OQ-32 (3b)): a new number of the one
        # global sequence, so it is behind every entry then at its price and ahead of every later arrival.
        ic = e.ice
        ic[3] += 1                                   # slice index k
        e.vis = min(ic[2], e.hidden)                 # peak, or the remainder (§4.2)
        e.hidden -= e.vis
        e.price = ic[0] + ic[3] * ic[1]              # base + k * ppd (§4.2 ICEwPI)
        e.seq = self.nxt()                           # a new timestamp at the activation event (§4.2, §5.2; E2)
        self.activations += 1
        ke_e = self.rested.get(e.oid)
        if ke_e is not None and ke_e[0] > 0 and ke_e[1] != 0:
            self.cov["continuation_slice"] += 1          # a rested remainder's next instance, b + (k_e + 1) d, ...

    def matchable(self, side, limit):
        # §4.2.1 Rule 1 / OQ-11: visible crossing quantity plus every hidden slice whose own price crosses.
        tot = 0
        for e in self.book:
            if e.side == side:
                continue
            if e.vis > 0 and crosses_in(side, limit, e.price):
                tot += e.vis
            if e.ice is not None:
                base, ppd, peak, k = e.ice
                rem = e.hidden
                while rem > 0:
                    k += 1
                    sz = min(peak, rem)
                    rem -= sz
                    if crosses_in(side, limit, base + k * ppd):
                        tot += sz
        return tot

    def _row(self, inc, e, f):
        return [inc.oid, e.oid, e.price, f, inc.oid] if inc.side == BUY else [e.oid, inc.oid, e.price, f, inc.oid]

    def choose(self, opp, inc):
        # §6.1: best price, then the earliest timestamp (lowest sequence number). Strict: no reading is left open.
        best = min(e.price for e in opp) if inc.side == BUY else max(e.price for e in opp)
        return min((e for e in opp if e.price == best), key=lambda e: e.seq)

    def submit(self, r):
        side, price, qty, oid, ex = int(r[C_SIDE]), int(r[C_PRICE]), int(r[C_QTY]), int(r[C_OID]), int(r[C_EXEC])
        if ex == FOK and self.matchable(side, price) < qty:
            self.cancel.setdefault(oid, []).append(["fok_kill", qty])
            self.end_submission(oid)
            return
        # Every incoming order aggresses with its whole quantity. An incoming iceberg (v2.14 §4.2.1 Rule 2 "Incoming
        # iceberg", amended 2026-10-20) is a chain of instances of size P (the last may be smaller): instance k trades
        # only at prices its own limit b + k d crosses (item (6)), a row ends where a resting entry ends or where an
        # incoming instance ends (item (2); OQ-33 revised), every row prices at the resting entry (item (3)), and the
        # sweep stops at the first instance that cannot trade, even if b would still cross (item (6); OQ-34 resolved).
        ice_in = ex == ICE
        peak, ppd = (int(r[C_PEAK]), int(r[C_PPD])) if ice_in else (0, 0)
        inc = Entry(side, price, qty, 0, oid, self.nxt(), None)
        swept = 0                                    # M: the quantity swept so far
        while inc.vis > 0:
            if ice_in:
                kk = swept // peak                   # the incoming instance in use
                lim = price + kk * ppd               # its own limit b + k d (item (6))
                room = (kk + 1) * peak - swept       # what is left of that instance (item (2))
            else:
                lim, room = price, inc.vis
            opp = [e for e in self.book if e.side != side and e.vis > 0 and crosses_in(side, lim, e.price)]
            if not opp:
                if ice_in and any(e.side != side and e.vis > 0 and crosses_in(side, price, e.price) for e in self.book):
                    self.cov["instance_limit_stop"] += 1      # stopped by its instance limit while b still crosses
                break
            e = self.choose(opp, inc)
            f = min(inc.vis, e.vis, room)
            row = self._row(inc, e, f)               # one row per (resting entry, incoming instance) pair
            if self.obs is not None:
                if self.k >= len(self.obs) or self.obs[self.k][4] != oid:
                    self.mismatch = {"reason": "the reference predicts a trade the engine did not write",
                                     "at_trade": self.k, "expected": row,
                                     "observed": self.obs[self.k] if self.k < len(self.obs) else None}
                    return
                if self.obs[self.k] != row:
                    self.mismatch = {"reason": "the trade row differs (price-time priority, price or quantity)",
                                     "at_trade": self.k, "expected": row, "observed": self.obs[self.k]}
                    return
                self.k += 1
            self.trades.append(row)
            if ice_in and f == room and f < min(inc.vis, e.vis):
                self.cov["instance_split_row"] += 1       # the row ended at an incoming instance boundary
            ke_e = self.rested.get(e.oid)
            if ke_e is not None and ke_e[0] > 0 and ke_e[1] != 0:
                self.cov["continuation_print"] += 1       # a later print on a remainder that kept its ladder place
            inc.vis -= f
            e.vis -= f
            swept += f
            if e.vis == 0:
                if e.hidden > 0:
                    self.activate(e)             # Rule 2 / Rule 4: matched in this same submission if it crosses
                else:
                    self.book.remove(e)
        left = inc.vis
        if ice_in and swept > 0:
            self.cov["incoming_icewpi_traded" if ppd != 0 else "incoming_ice_traded_d0"] += 1
        if left > 0:
            if ex == NON:
                self.book.append(inc)                # §6.2 item 4: the entry number it arrived with (§5.2)
            elif ex == ICE:
                # Rule 2 (4): the remainder R rests as an iceberg with total R and keeps its place in the ladder: it
                # continues as instance k_e = floor(M / P) at limit b + k_e d, shows v1 = min(P - u, R), u = M - k_e P
                # (the unused part of instance k_e), and keeps R - v1 hidden. No new timestamp (§5.2): its arrival
                # number. Its later slices continue the ladder, b + (k_e + 1) d, ... (OQ-34 resolved; no restart).
                ke = swept // peak
                u = swept - ke * peak
                inc.vis = min(peak - u, left)
                inc.hidden = left - inc.vis
                inc.price = price + ke * ppd
                inc.ice = [price, ppd, peak, ke]
                self.book.append(inc)
                self.rested[oid] = (ke, ppd)
                if u > 0:
                    self.cov["remainder_u_pos_icewpi" if ppd != 0 else "remainder_u_pos_d0"] += 1
                if ke > 0 and ppd != 0:
                    self.cov["remainder_off_base"] += 1       # rests at b + k_e d, not at b
            elif ex == IOC:
                self.cancel.setdefault(oid, []).append(["ioc_remainder", left])   # §6.2 item 5
            else:
                self.internal.append(f"oid {oid}: an accepted FOK kept {left} (pre-check and walk disagree)")
        self.end_submission(oid)

    def end_submission(self, oid):
        if self.obs is not None and self.mismatch is None and self.k < len(self.obs) and self.obs[self.k][4] == oid:
            self.mismatch = {"reason": "the engine wrote a trade the reference does not predict", "at_trade": self.k,
                             "expected": None, "observed": self.obs[self.k]}
        for e in self.book:
            if e.vis <= 0 and e.vis + e.hidden > 0:
                self.internal.append(f"after oid {oid}: oid {e.oid} rests with no shown slice in the reference")
        bids = [e.price for e in self.book if e.side == BUY and e.vis > 0]
        asks = [e.price for e in self.book if e.side == SELL and e.vis > 0]
        if bids and asks and max(bids) >= min(asks):
            self.internal.append(f"after oid {oid}: the reference book is crossed ({max(bids)} >= {min(asks)})")
        if self.after is not None and self.mismatch is None:
            self.after(oid, self)

    def run(self):
        for r in self.rows:
            self.submit(r)
            if self.mismatch is not None:
                return self
        if self.obs is not None and self.k < len(self.obs):
            self.mismatch = {"reason": "the engine wrote trades after the reference's last", "at_trade": self.k,
                             "expected": None, "observed": self.obs[self.k:]}
        return self

    def remaining(self):
        out = {}
        for e in self.book:
            if e.vis + e.hidden > 0:
                out[str(e.oid)] = out.get(str(e.oid), 0) + e.vis + e.hidden
        return out

    def shown(self):
        # W's book as the engine must show it after a submission: oid -> [side, [price], visible, hidden].
        return {str(e.oid): [e.side, [e.price], e.vis, e.hidden] for e in self.book if e.vis + e.hidden > 0}


def walk(tape):
    # W alone, with no engine run.
    return RefWalk(tape).run()


def observed_trades(rec):
    # The engine's trade rows in W's form; the submission is named by the order id it submitted.
    sub_oid = {int(a.submission): int(o) for o, a in rec.accepted.items()}
    for rj in rec.rejections:
        sub_oid.setdefault(int(rj.submission), int(rj.oid))
    return [[int(f.buyer_oid), int(f.seller_oid), int(f.price_int), int(f.qty_int), sub_oid.get(int(f.submission), -1)]
            for f in rec.fills]


def engine_shown(snap, ice_oids):
    # The engine's book after a submission, in W's form: oid -> [side, sorted shown prices, visible, hidden].
    # Visible rows from BookSnapshot.resting; hidden quantity from BookSnapshot.hidden_qty for every iceberg so far.
    out = {}
    for sd in (BUY, SELL):
        for row in snap.resting(sd):
            x = out.setdefault(str(int(row[2])), [sd, set(), 0, 0])
            x[1].add(int(row[0]))
            x[2] += int(row[1])
    for o in ice_oids:
        h = int(snap.hidden_qty(o))
        if h > 0:
            out.setdefault(str(o), [None, set(), 0, 0])[3] += h
    return {o: [x[0], sorted(x[1]), x[2], x[3]] for o, x in out.items()}


def judge_reference(tape, rec):
    # (W) against one engine record. Returns (ok, problems, info); problems are recorded in full on failure.
    rows = real_rows(tape)
    oids = [int(r[C_OID]) for r in rows]
    is_ice = {int(r[C_OID]): int(r[C_EXEC]) == ICE for r in rows}
    obs = observed_trades(rec)
    sub_of = {int(o): int(a.submission) for o, a in rec.accepted.items()}
    seen_ice, book_bad, not_shown, n_checked = [], [], [], [0]

    def after(oid, w):
        # The book check after every submission (finding F6, "slice shown"): each order W holds is shown at W's
        # price with W's visible quantity and holds W's hidden quantity, and nothing else is shown or hidden; in
        # particular every iceberg with remaining quantity shows a slice (§4.2.1 Rule 4; OQ-20 correction note (b)).
        if is_ice.get(oid):
            seen_ice.append(oid)
        s = sub_of.get(oid)
        if s is None or not 0 <= s < len(rec.post_snapshots):
            return                                   # a rejected order: recorded under "acceptance"
        n_checked[0] += 1
        got = engine_shown(rec.post_snapshots[s], seen_ice)
        for o, x in got.items():
            if x[2] == 0 and x[3] > 0:
                not_shown.append({"after_oid": oid, "oid": int(o), "hidden": x[3]})
        want = w.shown()
        if got != want:
            diff = sorted(set(got) | set(want), key=int)
            book_bad.append({"after_oid": oid, "submission": s,
                             "expected": {o: want.get(o) for o in diff if want.get(o) != got.get(o)},
                             "observed": {o: got.get(o) for o in diff if want.get(o) != got.get(o)}})

    w = RefWalk(tape, observed=obs, after=after).run()
    problems = {}
    acc = sorted(int(o) for o in rec.accepted)
    if acc != sorted(oids) or len(rec.rejections):
        problems["acceptance"] = {"expected_accepted": sorted(oids), "observed_accepted": acc,
                                  "observed_rejected": sorted(int(r.oid) for r in rec.rejections)}
    if w.mismatch is not None:
        problems["trade_sequence"] = dict(w.mismatch, expected_trades=walk(tape).trades, observed_trades=obs)
    else:
        exp_c = {str(o): sorted(cs) for o, cs in w.cancel.items()}
        obs_c = {str(int(o)): sorted([str(c.cause), int(c.qty_int)] for c in cs)
                 for o, cs in rec.cancelled_or_expired.items() if len(cs)}
        if exp_c != obs_c:
            problems["cancelled"] = {"expected": exp_c, "observed": obs_c}
        exp_r = w.remaining()
        obs_r = {}
        for o in oids:
            q = int(rec.still_resting.get(o, 0)) + int(rec.hidden_remaining.get(o, 0))
            if q:
                obs_r[str(o)] = q
        if exp_r != obs_r:
            problems["remaining"] = {"expected": exp_r, "observed": obs_r}
    if not_shown:
        problems["slice_shown"] = not_shown          # F6: an iceberg with remaining quantity and no shown slice
    if book_bad:
        problems["book_after_submission"] = {"n_submissions": len(book_bad),
                                             "after_oids": [x["after_oid"] for x in book_bad],
                                             "first": book_bad[0]}
    ev = {str(int(o)): int(q) for o, q in rec.evicted.items() if int(q)}
    if ev:
        problems["evicted"] = {"expected": {}, "observed": ev}
    if w.internal:
        problems["reference_internal"] = list(w.internal)
    return (not problems), problems, {"activations": w.activations, "book_checks": n_checked[0],
                                      "n_trades": len(obs), "volume": sum(t[3] for t in obs)}


# ---- (M) relation variants: functions of the tape and W only --------------------------------------------------------
ROTATION = ("M1", "M2", "M3", "M5", "M6")
AGENT_MAP = {1: 2, 2: 3, 3: 4, 4: 1}


def as_tape(rows):
    rows = np.asarray(rows, dtype=np.int32).reshape(-1, synth.N_COLS)
    return synth.pad_to(rows, len(rows) + 8)


def reach(row):
    # The most aggressive price any slice of the order can carry.
    p = int(row[C_PRICE])
    if int(row[C_EXEC]) == ICE and int(row[C_PPD]) != 0:
        last = p + (n_slices(row) - 1) * int(row[C_PPD])
        return max(p, last) if int(row[C_SIDE]) == BUY else min(p, last)
    return p


def can_cross(a, b):
    if int(a[C_SIDE]) == int(b[C_SIDE]):
        return False
    buy, sell = (a, b) if int(a[C_SIDE]) == BUY else (b, a)
    return reach(buy) >= reach(sell)


def traded_on_arrival(w):
    return {t[4] for t in w.trades}


def first_aggressor(rows, w, types_=(NON, IOC), qty_ok=lambda q: True):
    hit = traded_on_arrival(w)
    for s, r in enumerate(rows):
        if int(r[C_OID]) in hit and int(r[C_EXEC]) in types_ and qty_ok(int(r[C_QTY])):
            return s
    return None


def plan_relation(name, tape, w, j, seed):
    # A variant for relation `name`, or None when it does not apply to this tape (F7-F9).
    rows = np.array(real_rows(tape), copy=True)
    if name == "M1":
        groups, start = [], 0
        for s in range(1, len(rows) + 1):
            if s == len(rows) or int(rows[s][C_TS]) != int(rows[start][C_TS]):
                if s - start >= 2:
                    groups.append((start, s))
                start = s
        hit = traded_on_arrival(w)
        elig = []
        for a, b in groups:
            g = rows[a:b]
            if any(int(r[C_OID]) in hit for r in g):
                continue
            if any(can_cross(g[x], g[y]) for x in range(len(g)) for y in range(x + 1, len(g))):
                continue
            elig.append((a, b))
        if not elig:
            return None
        var = rows.copy()
        for a, b in elig:
            var[a:b] = rows[a:b][::-1]
        ties = [int(r[C_OID]) for a, b in elig for r in rows[a:b]]
        return {"name": "M1", "base": None, "var": as_tape(var), "args": {"tie_oids": ties}}
    if name == "M6":
        var = rows.copy()
        var[:, C_AGENT] = [AGENT_MAP.get(int(a), int(a)) for a in rows[:, C_AGENT]]
        return {"name": "M6", "base": None, "var": as_tape(var), "args": {"agent_map": dict(AGENT_MAP)}}
    if name in ("M2", "M3"):
        s = first_aggressor(rows, w)
        if s is None:
            return None
        base = rows[: s + 1].copy()
        var = base.copy()
        oid = int(base[s][C_OID])
        if name == "M2":
            if (j // len(ROTATION)) % 2 == 0:
                rel = "size_increased"
                var[s][C_QTY] = min(2 * int(base[s][C_QTY]), QTY_INT_MAX)
                if int(var[s][C_QTY]) == int(base[s][C_QTY]):
                    return None
            else:
                rel = "limit_tightened"
                d = 1 + int(seed) % 3
                var[s][C_PRICE] = int(base[s][C_PRICE]) - d if int(base[s][C_SIDE]) == BUY else int(base[s][C_PRICE]) + d
                if not 0 <= int(var[s][C_PRICE]) <= PRICE_INT_MAX:
                    return None
            return {"name": "M2", "base": as_tape(base), "var": as_tape(var),
                    "args": {"aggressor_oid": oid, "relation": rel}}
        var[s][C_EXEC] = IOC if int(base[s][C_EXEC]) == NON else NON
        return {"name": "M3", "base": as_tape(base), "var": as_tape(var),
                "args": {"oid": oid, "base_is_ioc": int(base[s][C_EXEC]) == IOC}}
    if name == "M5":
        s = first_aggressor(rows, w, qty_ok=lambda q: (q >= 2 and q % 2 == 0) or (q >= 3 and q % 3 == 0))
        if s is None:
            return None
        base = rows[: s + 1].copy()
        q = int(base[s][C_QTY])
        k = 2 if q % 2 == 0 else 3
        top = int(rows[:, C_OID].max())
        parts = []
        for i in range(k):
            p = base[s].copy()
            p[C_QTY] = q // k
            p[C_OID] = top + 1 + i
            parts.append(p)
        var = np.concatenate([base[:s], np.asarray(parts, dtype=np.int32)], axis=0)
        return {"name": "M5", "base": as_tape(base), "var": as_tape(var),
                "args": {"whole_oid": int(base[s][C_OID]), "split_oids": [top + 1 + i for i in range(k)]}}
    raise ValueError(name)


# ---- the band (reported, never asserted) -------------------------------------------------------------------------------
SIZE_MU, SIZE_SIGMA, SIZE_SCALE, SIZE_LO, SIZE_HI = 1.6, 0.7, 10, 1, 600   # the generator's size law (finding F12)
BAND_LAMBDA, BAND_Z = 0.05, 3.0                                             # pre-registered, never widened


def _phi(z):
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))


def size_moments():
    def cdf(x):        # P(SIZE_SCALE * X < x)
        return 0.0 if x <= 0 else _phi((math.log(x / SIZE_SCALE) - SIZE_MU) / SIZE_SIGMA)
    m1 = m2 = 0.0
    for k in range(SIZE_LO, SIZE_HI + 1):
        lo = 0.0 if k == SIZE_LO else cdf(k - 0.5)
        hi = 1.0 if k == SIZE_HI else cdf(k + 0.5)
        m1 += k * (hi - lo)
        m2 += k * k * (hi - lo)
    return m1, m2


MU_Q, MU_Q2 = size_moments()


def band(cell, n_effs):
    k = len(n_effs)
    nbar = sum(n_effs) / k
    c = cell.cross
    sd = math.sqrt(max(nbar * (c * MU_Q2 - c * c * MU_Q * MU_Q), 0.0) / k)
    return max(0.0, BAND_LAMBDA * c * nbar * MU_Q - BAND_Z * sd), c * nbar * MU_Q + BAND_Z * sd


def n_eff(draw):
    s = draw.params.get("safe_from_index")
    return int(draw.params["n_orders"]) if s is None else int(s)


# ---- what W says about the samples (no engine) ----------------------------------------------------------------------------
print(f"size law: mu_q = {MU_Q:.3f} ticks ({MU_Q * CONFIG.qty_tick:.3f} MW), E[q^2] = {MU_Q2:.1f}")
_rows = ["| mix | trades (W) | volume (W, ticks) | IOC cancelled | FOK killed | resting orders | band for one draw [lo, hi] | applicable relations |",
         "|---|---|---|---|---|---|---|---|"]
for mx, (cell, d) in SAMPLES.items():
    w = walk(d.tape)
    lo, hi = band(cell, [n_eff(d)])
    rel = [m for m in ROTATION if plan_relation(m, d.tape, w, 0, d.seed) is not None]
    nf = sum(1 for r in real_rows(d.tape) if int(r[C_EXEC]) == FOK)
    _rows.append(f"| {mx} | {len(w.trades)} | {sum(t[3] for t in w.trades)} | "
                 f"{sum(1 for cs in w.cancel.values() for c in cs if c[0] == 'ioc_remainder')} | "
                 f"{sum(1 for cs in w.cancel.values() for c in cs if c[0] == 'fok_kill')} of {nf} | "
                 f"{len(set(e.oid for e in w.book))} | [{lo:.0f}, {hi:.0f}] | {', '.join(rel) + (', M4' if nf else '')} |")
display(Markdown("\n".join(_rows)))
_cell, _d = SAMPLES["all_icewpi"]
print(f"W's first trades on the all_icewpi sample ([buyer, seller, price_int, qty_int, submitting oid]): {walk(_d.tape).trades[:8]}")
_cov = Counter()
for mx, (cell, d) in SAMPLES.items():
    _cov.update(walk(d.tape).cov)
print(f"W's incoming-iceberg coverage on the five samples (finding F21): {dict(sorted(_cov.items()))}")


## 6. Run

### The engine, jitted on every draw and variant, eager on the eager draws

> **Wired at K3.** The developer owns this section and edits **nothing else** (independence rule 2): sections 1–5 and 7 are the expectation. A disagreement with the expectation is escalated as a `SCOPE-CHANGE` candidate with both readings, never resolved by editing sections 5 or 7.

Section 7 consumes exactly **one name** from this section. If it were not defined, section 7 would record a failing `section 6 wired` check, run no draw, and the notebook would end `NOT CERTIFIED`.

1. **`run_session(tape, *, market_config, capacity_bounds, jit) -> RunRecord`**: the battery's record builder on one tape, exactly
   `tests.omiecid.reference.omie_invariants.build_record(tape, market_config=market_config, capacity_bounds=dict(capacity_bounds), jit=jit)`,
   with no other argument: one fresh book of an OPEN contract (the tape's contract id), the default `OMIEProductConfig()`, no cancellation, no gate expiry. `tape` is an `(N, 13)` int32 array (a draw's tape, a prefix of it or a relation variant, always right-padded with the loader's pad row); `market_config` is one of the four `MARKET_OF` configurations of section 3 (book depth 100 or 60 × iceberg slots 1 024 or 64, every other field default); `capacity_bounds` is the tape's `tape_capacity_bounds`. `jit=False` runs eager (`jax.disable_jit()`), `jit=True` jitted. Every call starts from its own fresh book; the function may raise, and section 7 records a raise as a failed check.

Section 7 calls `run_session(..., jit=True)` on every draw, every relation variant and every shrink candidate, and `run_session(..., jit=False)` on the eager draws (section 3). It reads these `RunRecord` fields: `accepted` (keys and each `submission`), `rejections` (each `submission`, `oid`), `fills` (each `buyer_oid`, `seller_oid`, `price_int`, `qty_int`, `submission`, `buyer_agent`, `seller_agent`), `still_resting`, `hidden_remaining`, `cancelled_or_expired` (each `cause`, `qty_int`), `evicted`, `observed_capacity`; and the whole record through `run_battery`, `check_m1_tie_permutation` … `check_m6_ownership_blindness` and `compare_records`.

In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_session(tape, *, market_config, capacity_bounds, jit) -> RunRecord,
# per the contract above.
#
# run_session: the battery's record builder on one tape, with the given pressure-level market
# configuration (one of MARKET_OF) and the tape's analytic capacity bounds, and no other
# argument: one fresh book of an OPEN contract (the tape's contract id), the default
# OMIEProductConfig(), no cancellation, no gate expiry. jit=False runs eager (build_record
# wraps the whole build in jax.disable_jit()), jit=True jitted. Every call builds its own
# fresh book; a raise propagates to section 7, which records it as a failed check.
# ---------------------------------------------------------------------------------------
from tests.omiecid.reference.omie_invariants import build_record


def run_session(tape, *, market_config, capacity_bounds, jit):
    """The battery's record builder on `tape`, with `market_config` and the tape's analytic bounds."""
    return build_record(tape, market_config=market_config, capacity_bounds=dict(capacity_bounds), jit=jit)


print("run_session wired: build_record(tape, market_config=MARKET_OF[(D, M)], "
      "capacity_bounds=tape_capacity_bounds(tape), jit)")


## 7. Certification

Per variation, per slot: draw (section 4), check the precondition, run jitted, judge with the battery (B), the reference walk (W) and the relations (M), compare eager where due (E), and fold the result into the certificate. No check stops the notebook: every assertion is evaluated and recorded, and a raise inside a draw is a failed check for that draw.

**Discards** (INVALID, not failed; re-drawn in the same cell from the slot's replacement seeds, and counted in the discard rate): the generator could not stay inside the cell's envelope (`CapacityBreach`), or the battery's I8 returned `invalid` on the draw's own run. A failed precondition or a run that raised is **also** re-drawn, but is recorded as a **failed** check first, so it can never pass silently.

**What is recorded** (compact, in full; finding F17):

1. One oracle assertion per (variation, cell), named by the cell, whose `observed` holds **one string per draw** in order: `seed|digest|battery|relations|W|trades,volume|flags` — the first 12 hex digits of `Draw.digest()`; one letter per check RB, I1–I9 and per relation M1–M6 (`p` pass, `f` fail, `n` not applicable or not run, `i` invalid, `x` variant outside the envelope); `p`/`f` for W; the engine's trade count and matched volume in ticks; flags `E+`/`E-` (eager equal / different), `S` (the draw holds a self-match). W's `p` covers the trade rows, the cancellations, the remaining quantities and the book after every submission, "slice shown" included (finding F6). A discarded slot is `seed|D|reason`. The assertion fails if any draw in the cell failed anything.
2. One oracle assertion per failing draw, `observed` holding **every** violation of every failing check (main run and variant), W's mismatch with the full expected and observed trade lists, every `slice_shown` breach and the book check's failing submissions (the first in full), the precondition problems, or the traceback.
3. The battery, folded per valid draw with `add_battery` (RB, I1–I9 of the draw's run and its variants combined, worst status first, then M1–M6), so the invariant table counts every check and lists failing seeds.
4. Aggregate assertions (section 1) and the per-variation digest (blake2b-128 over the valid draws' digests, in order).

**Shrinking** (finding F14) runs after the sweep, on the first `SHRINK_MAX_DRAWS` failing draws: the target is the first failing check of the draw (W before the battery, then the relation); the tape is bisected on its prefix and then thinned one row at a time while the target still fails; the result is frozen and recorded, and an assertion records whether it reproduces the target.

**Coverage of the incoming-iceberg branch** (finding F21): per variation, the number of valid draws whose W walk holds a trading incoming ICEwPI, an instance-limit stop, an instance-split row, a remainder at $u > 0$ and a continued-ladder print. On a variation whose mixes hold icebergs each must be at least 1 (an asserted check); the counts, with their occurrence totals, are reported.

**Reported, not asserted:** the band per cell and its summary, discard rates per variation and pressure level, the capacity peaks per pressure level, the slices W activated and the book states it checked, the largest slice count, and the relation application counts.

In [ ]:
import hashlib
import time as _time
import traceback
from collections import defaultdict

from tests.omiecid.reference import omie_invariants as inv

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Randomised fuzz across the parameter space",
    known_result="zero exceptions across every draw: RB and I1-I9 never fail (not_applicable is allowed; I8 invalid "
                 "discards); the engine's trade sequence, cancellations and remaining quantities equal the independent "
                 "price-time reference walk W (strict: a slice is activated eagerly, queued at its activation event and "
                 "shown in the book after every submission, F6; an incoming iceberg sweeps its total quantity as a "
                 "chain of instances of size P, instance k only where its limit b + k d crosses, stopping at the "
                 "first instance that cannot trade, one row per (resting entry, incoming instance), the remainder "
                 "resting as instance k_e = floor(M/P) at b + k_e d showing min(P - u, R), its ladder continued; "
                 "v2.14 §4.2.1 Rule 2 amended 2026-10-20, OQ-33, OQ-34); "
                 "M1-M6 hold on their variants, which also pass the battery and W; eager == jitted on the eager draws; "
                 "plus a pre-registered closed-form band on aggregate matched volume, reported and never a pass criterion",
    oracle_kind="invariant / distributional",
    quotes=QUOTES, citations=CITATIONS, constructions=(CONSTRUCTION,), requested_draws=REQUESTED,
    runtime_forecast_s=float(sum({"a": 120, "b": 720, "c": 18000}[v] for v in RUN_VARIATIONS)),
    started_at=T0,
    downstream="every scenario's reading of §4.5, §6.1, §6.2 and §6.6 under interaction (C02-C10, C13); the RL "
               "environment, which runs the same engine on sessions of this shape; the P3-PERF-REGEN-MASK-1 "
               "regression debt (the frozen fixtures)",
)

R_WIRE = "independence rule 2: K3 wires section 6"
R_CEIL = "the capacity-validity precondition: 100 per side, 100 trades, 200 iterations, 1024 icebergs, 64 slices"
R_PRESS = "review §5 item 3: book_depth and max_icebergs are independent axes, bounded independently (finding F2)"
R_DATA = "dataset precondition (section 4): §2.3, §4.2 (derived peak rule), §13.6 item 1, slice margin, envelope"
R_RUN = "the run completes (a raise is a failure)"
R_DRAW = ("zero exceptions: battery RB, I1-I9; W (§4.2, §4.2.1 Rules 1-4 incl. the incoming iceberg as an instance "
          "chain (Rule 2 items (1)-(6), Rule 3 ICE aggressor, amended 2026-10-20) and eager activation, §4.5, §5.2, "
          "§6.1, §6.2, OQ-11, OQ-31, OQ-32 (3b), OQ-33, OQ-34); M1-M6 (§6.6)")
R_COV = ("I9 non-vacuity of the amended incoming-iceberg branch: §4.2.1 Rule 2 items (2), (4), (6) are exercised "
         "(finding F21)")
COV_KEYS = ("incoming_icewpi_traded", "instance_limit_stop", "instance_split_row", "remainder_u_pos",
            "continuation_print")
R_ACCT = "every requested draw ran; the sweep is systematic (scenario-ladder §C15 step 1)"
R_REL = "M1-M6 are evaluated at scale (scenario-ladder §C15; findings F7-F10)"
R_EAGER = "eager == jitted (scenario-ladder: what eager and jitted mean here); finding F13"
R_SHRINK = "the shrinker (scenario-ladder §C15): the smallest session that still violates the same check"

CHECK_IDS = ("RB", "I1", "I2", "I3", "I4", "I5", "I6", "I7", "I8", "I9")
M_IDS = ("M1", "M2", "M3", "M4", "M5", "M6")
CODE = {"pass": "p", "fail": "f", "not_applicable": "n", "invalid": "i"}

# ---- wiring and configuration ----------------------------------------------------------------------------------------
WIRED = callable(globals().get("run_session"))
cert.check("section 6 wired: run_session is defined", WIRED, expected=True, observed=WIRED, unit="bool", rule=R_WIRE,
           detail="" if WIRED else "run_session is undefined: no draw is run")
cert.check_equal("the five configured ceilings equal the documented ones", list(CEILINGS.values()),
                 DOCUMENTED_CEILINGS, unit="count", rule=R_CEIL)
_press = {f"D={d},M={m}": {"book_depth": MARKET_OF[(d, m)].book_depth, "max_icebergs": MARKET_OF[(d, m)].max_icebergs,
                           "resting_margin": ENVELOPE_OF[(d, m)].max_resting_per_side,
                           "slot_margin": ENVELOPE_OF[(d, m)].max_concurrent_icebergs}
          for (d, m) in PRESSURES}
_press_ok = all(
    0 < ENVELOPE_OF[p].max_resting_per_side < MARKET_OF[p].book_depth
    and 0 < ENVELOPE_OF[p].max_concurrent_icebergs < MARKET_OF[p].max_icebergs
    and ENVELOPE_OF[p].max_trades < MARKET_OF[p].nTrades
    and ENVELOPE_OF[p].max_hops < MARKET_OF[p].max_match_iterations
    and MARKET_OF[p].nTrades == DEFAULT_MARKET.nTrades
    and MARKET_OF[p].max_match_iterations == DEFAULT_MARKET.max_match_iterations
    for p in PRESSURES) and len(set(PRESSURES)) == 4
cert.check("pressure configurations: two independent axes, every margin strictly below its configured ceiling",
           _press_ok, expected="margins < ceilings; trades and iterations at their defaults", observed=_press,
           unit="other", rule=R_PRESS)


# ---- helpers -----------------------------------------------------------------------------------------------------------------
def run(tape, cell, jit=True):
    return run_session(tape, market_config=MARKET_OF[(cell.depth, cell.slots)],
                       capacity_bounds=synth.tape_capacity_bounds(tape), jit=jit)


def judge(tape, cell, rec):
    w0 = walk(tape)
    rep = inv.run_battery(rec, expects_trade=len(w0.trades) > 0, envelope=ENVELOPE_OF[(cell.depth, cell.slots)])
    ok, problems, info = judge_reference(tape, rec)
    return w0, rep, ok, problems, info


def na(cid, reason):
    return inv.CheckResult(check_id=cid, status="not_applicable", violations=(), unobserved=(),
                           evidence={"reason": reason})


def combine(cid, results):
    # Worst status first: fail > pass > not_applicable (an invalid variant never reaches here).
    unobs = tuple(sorted({str(u) for r in results for u in r.unobserved}))
    bad = [r for r in results if r.status == "fail"]
    if bad:
        return inv.CheckResult(cid, "fail", tuple(v for r in bad for v in r.violations), unobs, {"combined": len(results)})
    status = "pass" if any(r.status == "pass" for r in results) else "not_applicable"
    return inv.CheckResult(cid, status, (), unobs, {"combined": len(results)})


def vjson(v):
    return {"check": v.check_id, "field": v.field, "message": v.message, "submission": v.submission, "oid": v.oid,
            "fill_row": v.fill_row, "expected": v.expected, "observed": v.observed}


def evaluate_relation(plan, cell, main_rec):
    # Run a relation's variant (and its base prefix); judge both with (B) and (W). (None, ...) = outside the envelope.
    env = ENVELOPE_OF[(cell.depth, cell.slots)]
    for t in (plan["base"], plan["var"]):
        if t is not None:
            try:
                synth.check_envelope(env, **synth.tape_capacity_bounds(t))
            except synth.CapacityBreach:
                return None, [], None, None
    base_rec = main_rec if plan["base"] is None else run(plan["base"], cell)
    var_rec = run(plan["var"], cell)
    sub = []
    for label, t, r in (("base", plan["base"], base_rec), ("variant", plan["var"], var_rec)):
        if t is None:
            continue
        _, rep, ok, problems, _ = judge(t, cell, r)
        if not rep.valid:
            return None, [], None, None
        sub.append((label, rep, ok, problems))
    name, a = plan["name"], plan["args"]
    if name == "M1":
        res = inv.check_m1_tie_permutation(base_rec, var_rec, tie_oids=a["tie_oids"])
    elif name == "M2":
        res = inv.check_m2_monotonicity(base_rec, var_rec, aggressor_oid=a["aggressor_oid"], relation=a["relation"])
    elif name == "M3":
        ioc, non = (base_rec, var_rec) if a["base_is_ioc"] else (var_rec, base_rec)
        res = inv.check_m3_ioc_non_equivalence(ioc, non, oid=a["oid"])
    elif name == "M5":
        res = inv.check_m5_split_aggressor(base_rec, var_rec, whole_oid=a["whole_oid"], split_oids=a["split_oids"])
    else:
        res = inv.check_m6_ownership_blindness(base_rec, var_rec, agent_map=a["agent_map"])
    return res, sub, base_rec, var_rec


def m4_result(tape, rec):
    foks = [int(r[C_OID]) for r in real_rows(tape) if int(r[C_EXEC]) == FOK]
    if not foks:
        return None
    return combine("M4", [inv.check_m4_fok_dichotomy(rec, fok_oid=o) for o in foks])


def self_matches(rec):
    return sum(1 for f in rec.fills if int(f.buyer_agent) == int(f.seller_agent))


# ---- the sweep -------------------------------------------------------------------------------------------------------------
def new_stats():
    return {"valid": 0, "slots_short": 0, "discard": Counter(), "rel": {m: Counter() for m in M_IDS},
            "rel_skipped": Counter(), "m6_selfmatch": 0, "eager": [], "activations": 0, "book_checks": 0,
            "peaks": {p: {} for p in PRESSURES}, "max_slices": 0, "digests": [], "volume": defaultdict(list),
            "qty_sum": 0, "qty_n": 0, "fail_draws": 0, "blocks": defaultdict(list), "cell_fail": set(),
            "cell_valid": Counter(), "ice_cov_draws": Counter(), "ice_cov_n": Counter()}


STATS = {v: new_stats() for v in RUN_VARIATIONS}
EAGER_DONE = Counter()
FAILING = []


def process(v, j, cell, seed):
    """One slot attempt. Returns True when the draw is valid (counted), False when it was discarded."""
    st = STATS[v]
    pkey = (cell.depth, cell.slots)
    try:
        d = make_draw(cell, seed)
    except synth.CapacityBreach as exc:
        cert.discard(seed, f"generation: {exc.axis} {exc.value} > {exc.limit}")
        st["discard"][("generation", pkey)] += 1
        st["blocks"][cell].append(f"{seed}|D|gen:{exc.axis}")
        return False
    problems = precondition(d, cell)
    if problems:
        cert.check(f"{LABEL[v]} seed {seed}: dataset precondition", False, seed=seed, expected="no problem",
                   observed=problems, unit="other", detail="; ".join(problems), rule=R_DATA)
        cert.discard(seed, "dataset precondition failed (recorded as a failed check)")
        st["discard"][("precondition", pkey)] += 1
        st["blocks"][cell].append(f"{seed}|D|precondition")
        st["cell_fail"].add(cell)
        return False
    try:
        rec = run(d.tape, cell)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{LABEL[v]} seed {seed}: the jitted run raised", False, seed=seed, expected="a RunRecord",
                   observed=type(exc).__name__, unit="other", detail=f"{exc!r} | {traceback.format_exc()}", rule=R_RUN)
        cert.discard(seed, f"run raised {type(exc).__name__} (recorded as a failed check)")
        st["discard"][("raise", pkey)] += 1
        st["blocks"][cell].append(f"{seed}|D|raise")
        st["cell_fail"].add(cell)
        FAILING.append({"v": v, "j": j, "seed": seed, "cell": cell, "tape": d.tape, "params": dict(d.params),
                        "target": "RAISE"})
        return False
    w0, rep, w_ok, w_prob, w_info = judge(d.tape, cell, rec)
    if not rep.valid:
        cert.add_battery(rep, seed=seed)                 # an I8 invalid draw: discarded by the accumulator
        i8 = next(r for r in rep.results if r.check_id == "I8")
        st["discard"][("I8", pkey)] += 1
        st["blocks"][cell].append(f"{seed}|D|I8:" + ";".join(i8.evidence.get("breaches", [])))
        return False

    # ---- a valid draw ----
    mres = {m: na(m, "not run on this draw") for m in M_IDS}
    rcode = {m: "n" for m in M_IDS}
    sub_reports = []
    applied_names = []
    m4 = m4_result(d.tape, rec)
    if m4 is not None:
        mres["M4"], rcode["M4"] = m4, CODE[m4.status]
        st["rel"]["M4"][m4.status] += 1
        applied_names.append("M4")
    applied = 0
    for t in range(len(ROTATION)):
        if applied >= int(RELATIONS_PER_DRAW):
            break
        name = ROTATION[(j + t) % len(ROTATION)]
        try:
            plan = plan_relation(name, d.tape, w0, j, seed)
            if plan is None:
                continue
            res, sub, base_rec, var_rec = evaluate_relation(plan, cell, rec)
        except Exception as exc:  # noqa: BLE001
            res = inv.CheckResult(name, "fail", (inv.Violation(name, "raised", f"{exc!r} | {traceback.format_exc()}"),),
                                  (), {})
            sub, base_rec, var_rec = [], None, None
        if res is None:
            rcode[name] = "x"
            st["rel_skipped"][name] += 1
            continue
        mres[name], rcode[name] = res, CODE[res.status]
        st["rel"][name][res.status] += 1
        sub_reports += [(f"{name}:{lab}", r_, ok_, pr_) for (lab, r_, ok_, pr_) in sub]
        if name == "M6" and self_matches(rec) > 0:
            st["m6_selfmatch"] += 1
        applied_names.append(name)
        applied += 1

    eflag = ""
    if cell.n == int(EAGER_LENGTH) and EAGER_DONE[cell.mix] < int(EAGER_PER_MIX):
        EAGER_DONE[cell.mix] += 1
        try:
            rec_e = run(d.tape, cell, jit=False)
            diff = inv.compare_records(rec_e, rec)
            cert.set_eager_jit(seed, diff == (), detail=f"mix {cell.mix}: " + (", ".join(diff) or "identical"))
            eflag = "E+" if diff == () else "E-"
        except Exception as exc:  # noqa: BLE001
            cert.check(f"eager == jitted (seed {seed})", False, kind="eager_jit", seed=seed, expected=True,
                       observed=False, unit="bool", detail=f"the eager run raised {exc!r}", rule=R_EAGER)
            eflag = "E!"
        st["eager"].append([cell.mix, seed, eflag])

    results = []
    for cid in CHECK_IDS:
        parts = [next(r for r in rep.results if r.check_id == cid)]
        parts += [next(x for x in r_.results if x.check_id == cid) for (_, r_, _, _) in sub_reports]
        results.append(combine(cid, parts))
    results += [mres[m] for m in M_IDS]
    cert.add_battery(inv.BatteryReport(results=tuple(results)), seed=seed)
    st["valid"] += 1
    st["cell_valid"][cell] += 1

    all_w_ok = w_ok and all(ok_ for (_, _, ok_, _) in sub_reports)
    st["activations"] += w_info["activations"]
    _cv = dict(w0.cov)                                                # finding F21: W alone, no engine reading
    _cv["remainder_u_pos"] = _cv.get("remainder_u_pos_d0", 0) + _cv.get("remainder_u_pos_icewpi", 0)
    for k_, n_ in _cv.items():
        st["ice_cov_n"][k_] += int(n_)
        st["ice_cov_draws"][k_] += int(n_ > 0)
    st["book_checks"] += w_info["book_checks"]
    cap = dict(rec.observed_capacity)
    cert.record_capacity({**cap, "hops": None})                      # hops: unobserved by the engine
    for k_, val in cap.items():
        st["peaks"][pkey][k_] = max(int(val), st["peaks"][pkey].get(k_, 0))
    st["max_slices"] = max(st["max_slices"], max_slices(d.tape))
    dig = d.digest()
    st["digests"].append(dig)
    st["volume"][cell].append((w_info["volume"], n_eff(d)))
    rr = real_rows(d.tape)
    st["qty_sum"] += int(rr[:, C_QTY].sum())
    st["qty_n"] += len(rr)

    bat = "".join(CODE.get(r.status, "?") for r in results[:len(CHECK_IDS)])
    rel = "".join(rcode[m] for m in M_IDS)
    flags = eflag + ("S" if self_matches(rec) else "")
    st["blocks"][cell].append(f"{seed}|{dig[:12]}|{bat}|{rel}|{'p' if all_w_ok else 'f'}|"
                              f"{w_info['n_trades']},{w_info['volume']}|{flags or '-'}")

    failed = [r for r in results if r.status == "fail"]
    if failed or not all_w_ok or eflag in ("E-", "E!"):
        st["fail_draws"] += 1
        st["cell_fail"].add(cell)
        detail = {
            "cell": cell._asdict(), "regime": d.params["regime"], "digest": dig, "relations_applied": applied_names,
            "failing_checks": {r.check_id: [vjson(x) for x in r.violations] for r in failed},
            "reference_walk": {"main": w_prob, **{lab: pr_ for (lab, _, ok_, pr_) in sub_reports if not ok_}},
            "eager": eflag or None,
        }
        cert.check(f"{LABEL[v]} seed {seed}: every check on the draw (violations in full)", False, seed=seed,
                   expected="no failing check; W equal on the draw and every variant", observed=detail, unit="other",
                   detail=f"failing: {[r.check_id for r in failed] + ([] if all_w_ok else ['W'])}", rule=R_DRAW)
        main_fail = [r.check_id for r in rep.results if r.status == "fail"]
        if not w_ok:
            target = "W"
        elif main_fail:
            target = main_fail[0]
        else:
            target = next((m for m in M_IDS if rcode[m] == "f"), None) or next(
                (lab.split(":")[0] for (lab, r_, ok_, _) in sub_reports
                 if not ok_ or any(x.status == "fail" for x in r_.results)), "EAGER")
        FAILING.append({"v": v, "j": j, "seed": seed, "cell": cell, "tape": d.tape, "params": dict(d.params),
                        "target": target})
    return True


T_SWEEP = _time.time()
if WIRED:
    for v in RUN_VARIATIONS:
        st, N, sweep = STATS[v], DRAWS_OF[v], SWEEP[v]
        t_v = _time.time()
        for j in range(N):
            cell = sweep[j % len(sweep)]
            got = False
            for r in range(int(REPLACEMENTS_PER_SLOT) + 1):
                if process(v, j, cell, slot_seed(v, j, r)):
                    got = True
                    break
            if not got:
                st["slots_short"] += 1
            if (j + 1) % 250 == 0 or j + 1 == N:
                print(f"{LABEL[v]}: {j + 1}/{N} slots, {st['valid']} valid, {sum(st['discard'].values())} discarded, "
                      f"{st['fail_draws']} failing draws, {(_time.time() - t_v):.0f} s")
print(f"sweep wall clock: {_time.time() - T_SWEEP:.0f} s")

In [ ]:
# ---- per-variation assertions: the per-cell records, accounting, coverage ------------------------------------------------
for v in RUN_VARIATIONS:
    st, N = STATS[v], DRAWS_OF[v]
    for idx, cell in enumerate(CELLS[v]):
        entries = st["blocks"].get(cell, [])
        cert.check(f"{LABEL[v]} cell {idx}: every draw clean",
                   cell not in st["cell_fail"] and st["cell_valid"][cell] > 0, seed=None,
                   expected="at least one valid draw, and every draw: no fail in RB, I1-I9, M1-M6; W equal; "
                            "precondition holds; run completes",
                   observed={"cell": cell._asdict(), "regime": REGIME[cell].name, "draws": entries}, unit="other",
                   detail=f"{st['cell_valid'][cell]} valid, {sum(1 for e in entries if '|D|' in e)} discarded",
                   rule=R_DRAW)
    vdig = hashlib.blake2b("".join(st["digests"]).encode("ascii"), digest_size=16).hexdigest()
    cert.check_equal(f"{LABEL[v]}: valid draws == requested", st["valid"], N, unit="count", rule=R_ACCT,
                     detail=f"slots short {st['slots_short']}; discards {dict((f'{a}@D={p[0]},M={p[1]}', n) for (a, p), n in st['discard'].items())}; "
                            f"variation digest {vdig}")
    empty = [list(c) for c in CELLS[v] if st["cell_valid"][c] == 0]
    cert.check(f"{LABEL[v]}: every cell of the sweep has a valid draw", not empty, expected=[], observed=empty,
               unit="other", rule=R_ACCT, detail=f"{len(CELLS[v]) - len(empty)} of {len(CELLS[v])} cells")
    has_fok = any(FOK in [int(t) for t in synth.FUZZ_TYPE_MIXES[c.mix]] for c in CELLS[v])
    need = [m for m in M_IDS if m != "M4" or has_fok]
    applied = {m: int(st["rel"][m]["pass"] + st["rel"][m]["fail"]) for m in M_IDS}
    short = {m: applied[m] for m in need if applied[m] < int(MIN_RELATION_APPLICATIONS)}
    cert.check(f"{LABEL[v]}: every relation applied at least {MIN_RELATION_APPLICATIONS} times", not short,
               expected={m: f">= {MIN_RELATION_APPLICATIONS}" for m in need}, observed=applied, unit="count",
               rule=R_REL, detail=f"short: {short}; skipped (variant outside the envelope): {dict(st['rel_skipped'])}; "
                                  f"not_applicable: { {m: int(st['rel'][m]['not_applicable']) for m in M_IDS} }")
    cert.check(f"{LABEL[v]}: M6 exercised a self-match (§6.6 at scale)", st["m6_selfmatch"] >= 1,
               expected=">= 1 M6 draw holding a self-match", observed=st["m6_selfmatch"], unit="count", rule=R_REL)
    if any(ICE in [int(t) for t in synth.FUZZ_TYPE_MIXES[c.mix]] for c in CELLS[v]):
        _need = [k_ for k_ in COV_KEYS if k_ not in ("incoming_icewpi_traded", "instance_limit_stop", "continuation_print")
                 or any(c.mix == "all_icewpi" for c in CELLS[v])]
        _got = {k_: int(st["ice_cov_draws"][k_]) for k_ in _need}
        cert.check(f"{LABEL[v]}: the amended incoming-iceberg branch is exercised (draws per case)",
                   all(n_ >= 1 for n_ in _got.values()), expected={k_: ">= 1" for k_ in _need}, observed=_got,
                   unit="count", rule=R_COV,
                   detail=f"occurrences: {dict(sorted(st['ice_cov_n'].items()))}; draws: "
                          f"{dict(sorted(st['ice_cov_draws'].items()))}")

_mixes_run = sorted({c.mix for v in RUN_VARIATIONS for c in CELLS[v]})
for mx in _mixes_run:
    cert.check(f"eager comparison present for mix {mx}", EAGER_DONE[mx] >= int(EAGER_PER_MIX), kind="eager_jit",
               expected=f">= {EAGER_PER_MIX}", observed=int(EAGER_DONE[mx]), unit="count", rule=R_EAGER)

# ---- reported, not asserted: the band, the discard rate, the peaks, the activation readings ----------------------------
_rows = ["| variation | cells with draws | cells in band | below | above | mean order size (ticks) vs mu_q | discard rate | slices activated (W) / book states checked | largest slice count |",
         "|---|---|---|---|---|---|---|---|---|"]
for v in RUN_VARIATIONS:
    st = STATS[v]
    outside, n_in = [], 0
    for cell, vals in st["volume"].items():
        lo, hi = band(cell, [n for _, n in vals])
        vbar = sum(x for x, _ in vals) / len(vals)
        if lo <= vbar <= hi:
            n_in += 1
        else:
            outside.append({"cell": list(cell), "draws": len(vals), "mean_volume": round(vbar, 3),
                            "band": [round(lo, 3), round(hi, 3)], "side": "below" if vbar < lo else "above"})
    nd = sum(st["discard"].values())
    rate = nd / (nd + st["valid"]) if nd + st["valid"] else 0.0
    mean_q = st["qty_sum"] / st["qty_n"] if st["qty_n"] else float("nan")
    _rows.append(f"| {LABEL[v]} | {len(st['volume'])} | {n_in} | {sum(o['side'] == 'below' for o in outside)} | "
                 f"{sum(o['side'] == 'above' for o in outside)} | {mean_q:.2f} vs {MU_Q:.2f} | {rate:.4f} | "
                 f"{st['activations']} / {st['book_checks']} | {st['max_slices']} |")
    cert.check(f"REPORTED, not asserted: {LABEL[v]} aggregate matched volume against the pre-registered band", True,
               expected={"lambda": BAND_LAMBDA, "z": BAND_Z, "mu_q": round(MU_Q, 6), "mu_q2": round(MU_Q2, 6)},
               observed={"cells": len(st["volume"]), "in_band": n_in, "outside": outside,
                         "mean_order_size": round(mean_q, 4) if st["qty_n"] else None,
                         "discard_rate": rate, "discards": {f"{a}@D={p[0]},M={p[1]}": n for (a, p), n in st["discard"].items()},
                         "peaks_per_pressure": {f"D={p[0]},M={p[1]}": pk for p, pk in st["peaks"].items()},
                         "slices_activated_in_W": st["activations"], "book_states_checked": st["book_checks"],
                         "largest_slice_count": st["max_slices"], "eager_draws": st["eager"],
                         "incoming_iceberg_coverage": {"draws": dict(sorted(st["ice_cov_draws"].items())),
                                                       "occurrences": dict(sorted(st["ice_cov_n"].items()))}},
               unit="qty_int", detail="a sanity signal only: never a pass criterion (card); the band is never widened",
               rule="scenario-ladder §C15: the pre-registered aggregate band")
display(Markdown("\n".join(_rows)))
for v in RUN_VARIATIONS:
    print(f"{LABEL[v]} incoming-iceberg coverage (F21), draws: {dict(sorted(STATS[v]['ice_cov_draws'].items()))}")
    print(f"{LABEL[v]} relations: " + ", ".join(f"{m} {dict(STATS[v]['rel'][m])}" for m in M_IDS))
    print(f"{LABEL[v]} capacity peaks per pressure level: "
          + "; ".join(f"D={p[0]},M={p[1]}: {pk}" for p, pk in STATS[v]["peaks"].items()))

In [ ]:
# ---- the shrinker (finding F14): on any violation, the smallest failing session, frozen ----------------------------------
def failing_labels(tape, cell, relations, j, seed):
    # Every failing label of one candidate tape: battery ids, "W", relation names, "RAISE"; or {"INVALID"}.
    try:
        synth.check_envelope(ENVELOPE_OF[(cell.depth, cell.slots)], **synth.tape_capacity_bounds(tape))
    except synth.CapacityBreach:
        return {"INVALID"}
    try:
        rec = run(tape, cell)
    except Exception:  # noqa: BLE001
        return {"RAISE"}
    w0, rep, ok, _, _ = judge(tape, cell, rec)
    if not rep.valid:
        return {"INVALID"}
    out = {r.check_id for r in rep.results if r.status == "fail"}
    if not ok:
        out.add("W")
    if "M4" in relations:
        m4 = m4_result(tape, rec)
        if m4 is not None and m4.status == "fail":
            out.add("M4")
    for name in relations - {"M4"}:
        try:
            plan = plan_relation(name, tape, w0, j, seed)
            if plan is None:
                continue
            res, sub, _, _ = evaluate_relation(plan, cell, rec)
        except Exception:  # noqa: BLE001
            out.add(name)
            continue
        if res is not None and (res.status == "fail" or any(
                not ok_ or any(x.status == "fail" for x in r_.results) for (_, r_, ok_, _) in sub)):
            out.add(name)
    return out


def shrink(item):
    target, cell, rows = item["target"], item["cell"], np.array(real_rows(item["tape"]), copy=True)
    rels = {target} if target in M_IDS else set()
    evals = [0]

    def fails(rws):
        if len(rws) == 0 or evals[0] >= int(SHRINK_MAX_EVALS):
            return False
        evals[0] += 1
        return target in failing_labels(as_tape(rws), cell, rels, item["j"], item["seed"])

    n = len(rows)
    if not fails(rows):
        return {"reproduced": False, "rows": rows, "n": n, "evals": evals[0], "labels": []}
    lo, hi = 1, n                                    # rows[:hi] fails
    while lo < hi:
        mid = (lo + hi) // 2
        if fails(rows[:mid]):
            hi = mid
        else:
            lo = mid + 1
    cur = rows[:hi]
    i = len(cur) - 2
    while i >= 0 and evals[0] < int(SHRINK_MAX_EVALS):
        cand = np.delete(cur, i, axis=0)
        if fails(cand):
            cur = cand
        i -= 1
    labels = sorted(failing_labels(as_tape(cur), cell, rels, item["j"], item["seed"]))
    return {"reproduced": target in labels, "rows": cur, "n": n, "evals": evals[0], "labels": labels}


SHRUNK = []
for k, item in enumerate(FAILING):
    v, seed, cell = item["v"], item["seed"], item["cell"]
    params = dict(item["params"], variation=LABEL[v], cell=cell._asdict(), target=item["target"])
    if k >= int(SHRINK_MAX_DRAWS) or item["target"] == "EAGER":
        cert.record_case(seed, item["tape"], dict(params, shrunk=False))
        continue
    try:
        out = shrink(item)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"shrinker: {LABEL[v]} seed {seed} ({item['target']})", False, seed=seed, expected=item["target"],
                   observed=type(exc).__name__, unit="other", detail=f"the shrinker raised {exc!r}", rule=R_SHRINK)
        cert.record_case(seed, item["tape"], dict(params, shrunk=False))
        continue
    tape_s = as_tape(out["rows"])
    path = SHRUNK_DIR / f"{CERT_ID}_shrunk_seed{seed}.npz"
    meta = dict(params, shrunk=True, rows_before=out["n"], rows_after=len(out["rows"]), labels=out["labels"])
    SHRUNK_DIR.mkdir(parents=True, exist_ok=True)
    np.savez(path, tape=tape_s, params_json=np.array(json.dumps(meta, sort_keys=True, default=str)))
    cert.record_case(seed, tape_s, meta)
    SHRUNK.append({"seed": seed, "target": item["target"], "rows": [out["n"], len(out["rows"])], "path": str(path)})
    cert.check(f"shrinker: {LABEL[v]} seed {seed} -> {len(out['rows'])} of {out['n']} rows still fails {item['target']}",
               out["reproduced"], seed=seed, expected=item["target"], observed=out["labels"], unit="other",
               detail=f"{out['evals']} evaluations; frozen to {path}; rows {canon(out['rows'].tolist())}", rule=R_SHRINK)
print(f"failing draws: {len(FAILING)}; shrunk: {SHRUNK}")

# ---- the certificate table (condensed: the per-cell records are summarised, every failure is listed in full) -------------
_p = cert.payload()
_cells = [a for a in _p["assertions"] if " cell " in a["name"] and a["name"].endswith("every draw clean")]
_other = [a for a in _p["assertions"] if a not in _cells]
lines = ["| check | kind | status | seed | expected | observed | detail |", "|---|---|---|---|---|---|---|"]
for a in _other:
    obs = canon(a["observed"])
    lines.append(f"| {a['name']} | {a['kind']} | {a['status'].upper()} | {a['seed']} | {canon(a['expected'])[:200]} | "
                 f"{obs if a['status'] == 'fail' else obs[:200]} | {a['detail'][:300]} |")
lines.append(f"| per-cell records ({len(_cells)}) | oracle | "
             f"{'PASS' if all(a['status'] == 'pass' for a in _cells) else 'FAIL'} | — | every draw clean | "
             f"{sum(a['status'] == 'fail' for a in _cells)} failing cell(s): "
             f"{[a['name'] for a in _cells if a['status'] == 'fail']} | one string per draw (results JSON) |")
lines.append("")
lines.append("| check | status | n_pass | n_fail | n_not_applicable | failing seeds | unobserved |")
lines.append("|---|---|---|---|---|---|---|")
for r in _p["invariants"]:
    lines.append(f"| {r['check_id']} | {r['status'].upper()} | {r['n_pass']} | {r['n_fail']} | {r['n_not_applicable']} | "
                 f"{r['failing_seeds']} | {', '.join(r['unobserved']) or '-'} |")
d_ = _p["draws"]
lines.append("")
lines.append(f"Draws: requested {d_['requested']}, valid {d_['valid']}, discarded {d_['discarded']} "
             f"(discard rate {d_['discard_rate']:.4f}). Capacity peaks: {_p['capacity']['peaks']}. "
             f"Failure kinds so far: {_p['failure_kinds'] or 'none'}.")
display(Markdown("\n".join(lines)))

## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")

In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))